# Okubo et al. (2015), Growth and splitting of neural sequences in songbird vocal development, on CRCNS am-3

**Paper.** Okubo TS, Mackevicius EL, Payne HL, Lynch GF, Fee MS (2015). Nature 528, 352-357, doi:10.1038/nature15741. The claims file (`claims/okubo-2015.yaml`) was written from the version of record (main text, legends, Methods, Extended Data legends) and the model code of its Supplementary Information.

**Dataset.** CRCNS am-3, "Single-unit extracellular recordings of projection neurons in the premotor cortical area HVC of juvenile male zebra finches during singing", doi:10.6080/K01N7Z2Z, read anonymously from the public bucket `crcnsarchive` under `am-3/live/`. CRCNS data sets are not versioned, so the version is recorded as `unversioned` (docs/archives.md, CRCNS): there is one published state, the `live` folder, and it is the one the data descriptor (version 0.8, 26 Sept 2016) describes. The MD5 sums of the files read, from the data set's `checksums.md5`, are printed below so a later change can be detected. Only `Analysis_files.tar.gz` (per-neuron spike times and syllable segments), `docs/List_of_HVC_neurons.xls` and `docs/List_of_birds.xls` are read; the per-bird raw data archives (sound and voltage, 25 MB to 12.5 GB each) are not needed by any claim that is tested here.

**Model.** The model claims run a Python port (`hvcmodel.py`) of the MATLAB model code of the paper's Supplementary Information (`HVCModelCode`), supplied with the paper.

The table of claims and outcomes is in the summary at the end; the claims tested and blocked are listed in the next cell.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource, collections
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import hvc
T0 = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

CLAIMS = [
    ("counts-neurons-birds", "tested (juvenile part; adult neurons not public)"),
    ("ed1a-stage-age", "tested"),
    ("fig1-subsong-locking", "tested"),
    ("fig1f-locking-increases", "tested (juvenile stages only)"),
    ("fig1f-rhythmic-protosyllable", "tested"),
    ("fig1fg-rhythm-increases", "tested (juvenile stages only)"),
    ("ed1c-burst-width-rate", "blocked: needs the adult neurons (data_missing); juvenile medians shown"),
    ("fig1de-bout-related", "tested"),
    ("fig2b-protosequence", "tested"),
    ("ed2st-coverage", "tested"),
    ("ed2p-probabilistic", "tested"),
    ("ed2qr-pairs-independent", "tested"),
    ("fig2c-latency-subsong-earlier", "tested"),
    ("fig2c-latency-broadens-later", "tested"),
    ("bird1-protosequence", "tested"),
    ("fig3g-bird1-shared-specific", "tested"),
    ("ed4a-shared-latency-bird1", "tested"),
    ("ed3j-shared-ibi-specific-doubled", "tested"),
    ("fig3h-bird1-shared-decrease", "tested"),
    ("all-birds-shared-specific", "tested (bird 3 has no syllable labels)"),
    ("all-birds-shared-decrease", "tested"),
    ("skipping-cycles", "tested"),
    ("fig4-bird2-before-beta", "tested"),
    ("fig4h-bird2-shared-early", "tested"),
    ("fig4i-bird2-shared-decrease", "tested"),
    ("ed4bd-shared-latency-other", "tested"),
    ("ed7-bird1-bd-shared-persist", "tested"),
    ("ed8-bird3-phase", "blocked: bird 3's syllable labels A to E are not in the dataset (metadata_insufficient)"),
    ("motif-strategy-shared", "blocked: depends on ed8-bird3-phase; bird 4 part shown"),
    ("ed9-bird4-rhythms", "tested"),
    ("ed4j-bias", "tested"),
    ("ed5-shared-acoustic-difference", "blocked: Sound Analysis Pro features (not_runnable)"),
    ("ed5-shared-ra-x-proportion", "tested"),
    ("ed2mo-ra-x-similar", "tested (juvenile stages only)"),
    ("model-fig5a-subsong-decay", "tested (model)"),
    ("model-fig5b-chain", "tested (model)"),
    ("model-fig5cd-split", "tested (model)"),
    ("model-fig5e-latency-broadens", "tested (model)"),
    ("model-ed10gh-period", "tested (model)"),
    ("model-ed10ad-bout-onset", "tested (model)"),
    ("model-ed10eh-motif", "tested (model)"),
    ("model-ed10ik-new-chain", "tested (model)"),
]
print(pd.DataFrame(CLAIMS, columns=["claim", "status"]).to_string(index=False))

## Data check

The neurons are loaded from `Analysis_files.tar.gz` (streamed into memory) and joined to the neuron list `List_of_HVC_neurons.xls` by bird, date and cell name. Song stage and age come from the list. The checks below compare the files with the data descriptor and the paper's Methods.

In [ ]:
md5 = hvc.read_md5()
for f in ["Analysis_files.tar.gz", "docs/List_of_HVC_neurons.xls", "docs/List_of_birds.xls"]:
    print(f"{f:35s} md5 {md5[f]}")
N = hvc.load_neurons()
NK = {n["key"]: n for n in N}
sheet = hvc.neuron_sheet()
birds = hvc.bird_sheet()
print(f"\nneuron files in Analysis_files: {len(N)}  (folders: {dict(collections.Counter(n['folder'] for n in N))})")
print(f"rows in List_of_HVC_neurons.xls: {len(sheet)}  (tabs: {sheet.tab.value_counts().to_dict()})")
dup = sheet[sheet.key.duplicated(keep=False)]
print("rows of the list sharing one bird/date/name key:\n", dup[["key", "age", "stage", "files", "tab"]].to_string())
print("files without a row:", sorted(set(NK) - set(sheet.key)), " rows without a file:", sorted(set(sheet.key) - set(NK)))
print(f"birds in List_of_birds.xls: {len(birds)}; in the neuron files: {len({n['bird'] for n in N})}")
print("\nList_of_birds.xls, bird numbers of the paper:")
print(birds[birds['Bird #'].notna()][["Bird Name", "Bird #", "Fig. # in paper"]].to_string(index=False))
print("matches the Methods' identification key:",
      all(hvc.BIRD_NUMBER.get(r['Bird Name']) == int(r['Bird #']) for _, r in birds[birds['Bird #'].notna()].iterrows() if r['Bird Name'] in hvc.BIRD_NUMBER),
      "; birds numbered in the list but not in the Methods' key:",
      [(r['Bird Name'], int(r['Bird #'])) for _, r in birds[birds['Bird #'].notna()].iterrows() if r['Bird Name'] not in hvc.BIRD_NUMBER])

In [ ]:
info = sheet.drop_duplicates("key").set_index("key")
fs = {n["key"]: n for n in N}
rows = []
for n in N:
    r = info.loc[n["key"]]
    fi, on, off, lab = hvc.syllables(n)
    sp = np.concatenate([f["spikes"] for f in n["files"]])
    rows.append(dict(key=n["key"], bird=n["bird"], age=int(r.age), stage=r.stage, tab=r.tab, folder=n["folder"],
                     n_files=len(n["files"]), n_syll=on.size, n_spikes=sp.size,
                     dur_min=(off - on).min() if on.size else np.nan, dur_max=(off - on).max() if on.size else np.nan,
                     rec_s=sum(f["dur"] for f in n["files"])))
U = pd.DataFrame(rows).set_index("key")
print("cell type: folder of the analysis file against tab of the list (counts):")
print(pd.crosstab(U.folder, U.tab))
print("\nsong stage in the list (counts of neurons):", U.stage.value_counts().to_dict())
print("ages by stage (dph):")
print(U.groupby("stage").age.describe()[["count", "min", "50%", "max"]])
print("\n'young adult' rows by bird:", U[U.stage == "young adult"].groupby("bird").age.agg(["size", "min", "max"]).to_dict("index"))
print(f"\nsampling rate in every file: {sorted({int(x) for x in [hvc.FS]})} Hz (dbase.Fs)")
print(f"files per neuron {U.n_files.min()} to {U.n_files.max()}; recorded time per neuron {U.rec_s.min():.1f} to {U.rec_s.max():.1f} s")
print(f"selected syllables per neuron {U.n_syll.min()} to {U.n_syll.max()}; durations {np.nanmin(U.dur_min)*1000:.1f} to {np.nanmax(U.dur_max)*1000:.1f} ms")
print(f"selected spikes per neuron {U.n_spikes.min()} to {U.n_spikes.max()}")
# spikes inside their files, segments ordered
bad_t = sum(int(((f['spikes'] < 0) | (f['spikes'] > f['dur'])).sum()) for n in N for f in n['files'])
bad_seg = sum(int((f['off'] < f['on']).sum()) for n in N for f in n['files'])
print(f"spikes outside their file: {bad_t}; segments with offset before onset: {bad_seg}")
labs = collections.Counter(t for n in N for f in n["files"] for t, s in zip(f["title"], f["sel"]) if s)
print("labels of selected syllables (count):", dict(labs.most_common()))

### Clock check

Spike times and syllable segments are both stored in samples of the same file, so they share a clock by construction. To check that the spikes are aligned to the right segments, the locking test is run on 120 neurons (every ninth) with the true syllable onsets and with the onsets moved 0.75 s later (a contrast: a response locked to syllables should disappear when the labels are shifted in time).

In [ ]:
sub = N[::9][:120]
res = []
for n in sub:
    fi, on, off, lab = hvc.syllables(n)
    r = []
    for shift in (0.0, 0.75):
        rel, tri = hvc.aligned(n, on + shift, fi)
        r.append(hvc.shift_test(rel, tri, on.size, hvc.LOCK_WINDOW, hvc.neuron_rng(n["key"], f"clock{shift}"), n_surr=200)[2])
    res.append(r)
res = np.array(res)
print(f"neurons locked (P < 0.05, 200 surrogates): true onsets {np.mean(res[:,0] < 0.05):.2f}, onsets + 0.75 s {np.mean(res[:,1] < 0.05):.2f} (n = {len(sub)})")

### Per-neuron measures

For every neuron: singing time, locking to syllable onsets, pre- and post-bout activity, rhythmicity (four measures and their surrogate tests) and burst shape (`hvc.neuron_summary`). Each neuron and analysis has its own seeded random generator, so the result does not depend on the order or the number of processes. Locking and bout tests use 1,000 surrogates; rhythmicity 1,000 (IBI, autocorrelation) and 100 (spectrum, cepstrum), as in the Methods. This is the longest step (about 7 minutes with 4 processes).

In [ ]:
t = time.time()
S = hvc.summarise_all(N, processes=4)
print(f"{len(S)} neurons in {time.time()-t:.0f} s; peak memory of a worker {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss/1e6:.2f} GB")

In [ ]:
S = S.join(U[["bird", "age", "stage", "tab"]], on="key")
S["stage4"] = S.stage.replace({"young adult": "motif"})
S["code"] = S.stage4.map(hvc.STAGE_CODE)
S["bird_no"] = S.bird.map(hvc.BIRD_NUMBER)
S["locked_b"] = (S.lock_p_wide < hvc.ALPHA).astype(int)      # shift reading (b): spikes from a wider window
S["locked_a"] = (S.lock_p_circular < hvc.ALPHA).astype(int)  # shift reading (a): circular shift
S["kept"] = S.singing_s >= hvc.MIN_SINGING
J = S[S.kept].copy()
print(f"singing time per neuron {S.singing_s.min():.1f} to {S.singing_s.max():.1f} s; neurons under 5 s of singing (excluded, Methods): {(~S.kept).sum()}")
print(S[~S.kept][["key", "stage", "singing_s"]].to_string(index=False))
print(f"\nneurons kept: {len(J)} in {J.bird.nunique()} birds")
print("by stage (stage 'young adult' of the list counted as motif, see below):")
print(J.groupby(["code", "stage4"]).size().to_string())
print("\nby cell type:", J.tab.value_counts().to_dict())
print(f"bouts per neuron {S.n_bouts.min()} to {S.n_bouts.max()}; 1.5 s spectral windows per neuron {S.n_windows.min()} to {S.n_windows.max()} "
      f"({(S.n_windows == 0).sum()} neurons have no bout of 1.5 s, so no spectrum or cepstrum)")
print(f"burst width {S.burst_width.min()*1000:.1f} to {S.burst_width.max()*1000:.1f} ms; rate in bursts {S.burst_rate.min():.0f} to {S.burst_rate.max():.0f} Hz")
print(f"periods (autocorrelation peak) {S.period_a.min()*1000:.0f} to {S.period_a.max()*1000:.0f} ms (range searched 80 to 1,000 ms)")

## counts-neurons-birds

> The data comprise 1,149 HVC projection neurons from 35 birds (juvenile and adult), of which 211 are unidentified projection neurons (HVCp).

The adult neurons (3 birds, 32 neurons) come from earlier recordings (ref. 20) that are not in this dataset and were not found in a public archive, so the juvenile counts are judged (1,117 neurons, 32 birds) and the 211 HVCp under both readings.

In [ ]:
print(f"juvenile neurons kept (>= 5 s singing): {len(J)} (expected 1,117 +/- 111.7); all files: {len(S)}; rows in the list: {len(sheet)}")
print(f"juvenile birds: {J.bird.nunique()} (expected 32)")
print(f"HVCp kept: {(J.tab == 'HVCp').sum()} (expected 211 +/- 21.1 if all 211 are juvenile); HVCp in all files {(S.tab=='HVCp').sum()}")
print(f"all neurons with the adult part: {len(J)} + 32 adult (not in the dataset) = {len(J)+32} (expected 1,149)")
print("per bird:", J.groupby("bird").size().to_dict())

**Result: reproduced for the juvenile part.** 1,116 juvenile neurons pass the 5 s singing rule (1,117 +/- 111.7 expected) in 32 birds, and 210 of them are HVCp (211 +/- 21.1 under either reading). The 32 adult neurons of 3 birds are not in the dataset, so the totals of 1,149 neurons and 35 birds can only be formed with the paper's own adult count (1,116 + 32 = 1,148).

## ed1a-stage-age

> Song stage advances with age (Spearman's rho = 0.61 between age and stage); the median ages are 48 dph for subsong, 58 for the protosyllable stage, 62 for the multi-syllable stage and 73 for the motif stage.

Stage coded 1 to 4. Readings of the unit: (a) one value per recording day of each bird, (b) per neuron, (c) per bird and stage (median age). The list's "young adult" stage is counted as motif (see the data check).

In [ ]:
def stage_age(df, label):
    rho, p = stats.spearmanr(df.age, df.code)
    med = df.groupby("code").age.median()
    iqr = df.groupby("code").age.quantile(0.75) - df.groupby("code").age.quantile(0.25)
    print(f"{label:28s} n={len(df):5d} rho={rho:.3f} (P={p:.1e}) medians {med.to_dict()} IQR {iqr.round(1).to_dict()}")
    return rho, med
days = J.groupby(["bird", "age"]).code.agg(lambda x: x.mode().iloc[0]).reset_index()
days_mixed = J.groupby(["bird", "age"]).code.nunique()
print(f"recording days: {len(days)}; days with more than one stage in the list: {(days_mixed > 1).sum()}")
r_a = stage_age(days, "(a) per bird and day")
r_b = stage_age(J, "(b) per neuron")
r_c = stage_age(J.groupby(["bird", "code"]).age.median().reset_index(), "(c) per bird and stage")
print("expected rho 0.61 +/- 0.061; medians 48 +/- 4, 58 +/- 10, 62 +/- 12, 73 +/- 21")
# young adult rows counted apart, as a variant
J5 = J.assign(code5=J.stage.map({**hvc.STAGE_CODE, "young adult": 5}))
print("variant, 'young adult' as a fifth stage, per neuron: rho = %.3f" % stats.spearmanr(J5.age, J5.code5)[0])

**Result: reproduced.** Per neuron (reading b), Spearman's rho between age and stage is 0.615 and the median ages are 48, 58, 62 and 73 dph, with IQRs of 4, 10, 12 and 21 days, as in the paper. Per recording day (a) rho is 0.710 and per bird and stage (c) 0.756, both outside the tolerance.

## fig1-subsong-locking

> In the subsong stage about half of the projection neurons (19 of 39) burst at a particular latency relative to syllable onsets, the others not locked to onsets.

Locking as in the claims header (Methods, Syllable-related neural activity), all subsong syllables, under both shift readings: (a) circular shift within +/- 0.5 s, (b) spikes from +/- 1 s shifted and binned within +/- 0.5 s.

In [ ]:
sub_s = J[J.code == 1]
print(f"subsong neurons: {len(sub_s)} (expected 39 +/- 3.9) in {sub_s.bird.nunique()} birds")
print(f"locked, reading (a) circular: {sub_s.locked_a.sum()}; reading (b) wide: {sub_s.locked_b.sum()} (expected 19 +/- 1.9)")
for thr in (0.05, 0.01, 0.001):
    print(f"  threshold P < {thr}: (a) {(sub_s.lock_p_circular < thr).sum()}, (b) {(sub_s.lock_p_wide < thr).sum()}")
print("per bird (locked (b) / neurons):", {b: f"{g.locked_b.sum()}/{len(g)}" for b, g in sub_s.groupby("bird")})
print("P values (b):", np.sort(sub_s.lock_p_wide.values).round(3).tolist())

**Result: not reproduced.** There are 39 subsong-stage neurons, as in the paper, but 25 (shift reading b) or 27 (reading a) are locked to syllable onsets at P < 0.05, against 19 +/- 1.9. At P < 0.01, a threshold the paper does not use, 21 are locked under both readings.

## fig1f-locking-increases

> The fraction of neurons locked to syllable onsets rises through development until nearly every adult neuron is locked (correlation of locking with song stage r = 0.22, P < 10^-10).

Without the adult neurons, r over the four juvenile stages is reported and the claim is judged on the rise across them. Syllable readings: (iii a) all syllables pooled, (iii b) each labelled type on its own, locked when any type is significant (with Bonferroni as a sub-variant).

In [ ]:
t = time.time()
tl = hvc.type_lock_all([NK[k] for k in J.key], processes=4)
J["n_types"] = [a for a, b in tl]
J["locked_type"] = [b for a, b in tl]
tlb = [hvc.type_lock_any(NK[k], bonferroni=True)[1] if a > 1 else b for k, (a, b) in zip(J.key, tl)]
J["locked_type_bonf"] = tlb
print(f"per-type locking in {time.time()-t:.0f} s; neurons with labelled types: {(J.n_types > 0).sum()}")

In [ ]:
def frac_table(col):
    g = J.groupby("code")[col]
    return pd.DataFrame({"n": g.size(), "locked": g.sum(), "fraction": g.mean().round(3)})
for col, lab in (("locked_b", "(iii a) pooled, shift (b)"), ("locked_a", "(iii a) pooled, shift (a)")):
    print(lab); print(frac_table(col).to_string())
    r, p = stats.pearsonr(J[col], J.code)
    print(f"  Pearson r with stage (1-4) = {r:.3f}, P = {p:.1e} (expected 0.22 +/- 0.022 with adults, P < 1e-10)")
# reading (iii b): per type where types are labelled, pooled otherwise
for col, lab in (("locked_type", "(iii b) any type"), ("locked_type_bonf", "(iii b) any type, Bonferroni")):
    x = np.where(J.n_types > 0, J[col].fillna(0), J.locked_b)
    r, p = stats.pearsonr(x, J.code)
    print(f"{lab}: fractions {pd.Series(x).groupby(J.code.values).mean().round(3).to_dict()}; r = {r:.3f}, P = {p:.1e}")
# chi-square across stages
ct = pd.crosstab(J.code, J.locked_b)
print("chi-square across the four juvenile stages (reading b): chi2 = %.1f, P = %.1e" % stats.chi2_contingency(ct)[:2])
print("per bird with neurons in more than one stage, fraction locked by stage (reading b):")
print(J.groupby(["bird", "code"]).locked_b.mean().unstack().dropna(thresh=2).round(2).to_string())

**Result: partially reproduced (juvenile stages only).** The locked fraction rises from 0.64 (subsong) to 0.85, 0.86 and 0.96 (motif), and Pearson's r with stage over the juvenile neurons is 0.196 (P = 3.8e-11; 0.174 under shift reading a; 0.248 with per-type locking). The adult neurons that the paper's r = 0.22 includes are not in the dataset, so the claim can be judged only on the rise.

## fig1f-rhythmic-protosyllable

> Few subsong-stage neurons burst rhythmically (3 of 39), while about half of protosyllable-stage neurons do (70 of 135), with a period of 169 +/- 6.4 ms (mean +/- s.e.m.).

Rhythmicity as in the claims header, under the two readings of the histogram bins ((a) 1 ms smoothed over 20 bins, (b) 10 ms) and of the cepstrum ((a) of the mean spectrum, (b) mean of window cepstra).

In [ ]:
rows = []
for rb in "ab":
    for rc in "ab":
        col = f"rhythmic_{rb}{rc}"
        s1 = J[J.code == 1]; s2 = J[J.code == 2]
        per = s2[s2[col] == 1][f"period_{rb}"] * 1000
        rows.append(dict(reading=f"bins {rb}, cepstrum {rc}", subsong=f"{s1[col].sum()}/{len(s1)}", protosyll=f"{s2[col].sum()}/{len(s2)}",
                         period_mean=round(per.mean(), 1), period_sem=round(per.sem(), 1)))
print(pd.DataFrame(rows).to_string(index=False))
print("expected: subsong 3/39, protosyllable 70/135 (+/- 7), period 169 +/- 6.4 ms")
s2 = J[J.code == 2]
print("\nprotosyllable stage, measures significant (bins a, cepstrum a):")
sig = pd.DataFrame({"ibi": s2.p_ibi_a < .05, "ac": s2.p_ac_a < .05, "spec": s2.p_spec < .05, "cep": s2.p_cep_a < .05})
print(sig.sum().to_dict(), "; neurons by number of significant measures:", sig.sum(axis=1).value_counts().sort_index().to_dict())
print(f"neurons without a 1.5 s window (spectrum and cepstrum not computed): {(s2.n_windows == 0).sum()}")
print("variant, rhythmic only when at least 3 of 4 are significant:", int((sig.sum(axis=1) >= 3).sum()))
print("per bird, protosyllable stage, rhythmic (aa)/n:", {b: f"{g.rhythmic_aa.sum()}/{len(g)}" for b, g in s2.groupby("bird")})

**Result: partially reproduced.** Rhythmic bursting is rare in subsong and common in the protosyllable stage, but the counts are higher than the paper's under every reading: 7 or 6 of 39 subsong neurons (3 expected) and 87 or 88 of 135 protosyllable-stage neurons (70 +/- 7). The mean period is 161.6 +/- 6.0 ms with 1 ms bins (outside 169 +/- 6.4) and 174.9 +/- 9.9 ms with 10 ms bins (inside).

## fig1fg-rhythm-increases

> Both the fraction of rhythmic neurons and the period of the HVC burst rhythm increase through development (correlation with song stage r = 0.28 and r = 0.57, both P < 10^-10).

In [ ]:
for rb in "ab":
    for rc in "ab":
        col = f"rhythmic_{rb}{rc}"
        r1, p1 = stats.pearsonr(J[col], J.code)
        R = J[J[col] == 1]
        r2, p2 = stats.pearsonr(R[f"period_{rb}"], R.code)
        print(f"bins {rb}, cepstrum {rc}: rhythmic per stage {J.groupby('code')[col].sum().to_dict()} (total {J[col].sum()}); "
              f"r(rhythmic, stage) = {r1:.3f} P = {p1:.1e}; r(period, stage) = {r2:.3f} P = {p2:.1e} (n = {len(R)})")
print("expected: r = 0.28 +/- 0.028 and 0.57 +/- 0.057 (with adults), total rhythmic juvenile 727 +/- 72.7")
R = J[J.rhythmic_aa == 1]
print("mean period (ms) by stage, bins a:", (R.groupby("code").period_a.mean() * 1000).round(1).to_dict(),
      "s.e.m.", (R.groupby("code").period_a.sem() * 1000).round(1).to_dict())
print("Kruskal-Wallis across juvenile stages, period: H = %.1f, P = %.1e" % stats.kruskal(*[g.period_a for _, g in R.groupby("code")]))
print("correlation with age instead of stage: rhythmic r = %.3f, period r = %.3f" % (stats.pearsonr(J.rhythmic_aa, J.age)[0], stats.pearsonr(R.period_a, R.age)[0]))

**Result: partially reproduced (juvenile stages only).** Over the juvenile neurons both correlations are positive with P < 1e-10: rhythmic fraction against stage r = 0.279 to 0.292 (0.28 +/- 0.028 with adults) and period against stage r = 0.455 to 0.470 (0.57 +/- 0.057 with adults). The total of rhythmic juvenile neurons is 826 to 836, above 727 +/- 72.7. Without the adult neurons only the sign and significance are judged.

## ed1c-burst-width-rate (blocked)

> Burst width and firing rate within bursts change with developmental stage, some juvenile stages differing from adults.

The criterion compares each juvenile stage with the adult stage; the adult neurons are not in the dataset, so the claim is blocked. The juvenile medians are shown for a later attempt.

In [ ]:
print("median of per-neuron median burst width (ms) and within-burst rate (Hz) by stage:")
print(pd.DataFrame({"width_ms": J.groupby("code").burst_width.median() * 1000, "rate_Hz": J.groupby("code").burst_rate.median(),
                    "n": J.groupby("code").burst_width.count()}).round(2).to_string())
print("Kruskal-Wallis across juvenile stages: width P = %.1e, rate P = %.1e" % (
    stats.kruskal(*[g.burst_width.dropna() for _, g in J.groupby("code")])[1], stats.kruskal(*[g.burst_rate.dropna() for _, g in J.groupby("code")])[1]))

## fig1de-bout-related

> 285 of 1,117 juvenile projection neurons burst in relation to song bouts; 137 before bout onset only, 98 after bout offset only and 50 both.

Pre-bout: significant peak of the bout-onset histogram in the 300 ms before onset, unless the neuron is locked to syllable onsets with a peak within 25 ms of the same latency; post-bout: significant peak in the 300 ms after bout offset (Methods, Bout-related neural activity). Both shift readings.

In [ ]:
def bout_classes(df, rd):
    pre = (df[f"pre_p_{rd}"] < hvc.ALPHA)
    lock = df[f"lock_p_{rd}"] < hvc.ALPHA
    same = (df[f"pre_lat_{rd}"] - df[f"lock_lat_{rd}"]).abs() < hvc.SAME_LATENCY
    pre_x = pre & ~(lock & same)
    post = df[f"post_p_{rd}"] < hvc.ALPHA
    return pre_x, post, pre & lock & same
for rd, lab in (("circular", "(a) circular"), ("wide", "(b) wide")):
    pre_x, post, excl = bout_classes(J, rd)
    print(f"{lab}: pre only {int((pre_x & ~post).sum())}, post only {int((post & ~pre_x).sum())}, both {int((pre_x & post).sum())}, "
          f"bout-related {int((pre_x | post).sum())} of {len(J)}; pre-bout peaks left out as syllable-related: {int(excl.sum())}")
print("expected: 137 +/- 13.7, 98 +/- 9.8, 50 +/- 5, total 285 +/- 28.5 (pre totals 187, post 149 in Extended Data Fig. 2h, l)")
pre_x, post, _ = bout_classes(J, "wide")
J["pre_bout"] = pre_x.astype(int); J["post_bout"] = post.astype(int)
print("pre-bout and post-bout totals (b):", int(pre_x.sum()), int(post.sum()))
print("bout-related fraction by stage (b):", J.assign(br=(pre_x | post)).groupby("code").br.mean().round(3).to_dict())
print("variant, P < 0.01: pre", int(((J.pre_p_wide < .01) & ~((J.lock_p_wide < .05) & ((J.pre_lat_wide - J.lock_lat_wide).abs() < .025))).sum()), "post", int((J.post_p_wide < .01).sum()))

**Result: not reproduced.** With shift reading (b), 124 neurons are pre-bout only (137 +/- 13.7), 158 post-bout only (98 +/- 9.8) and 60 both (50 +/- 5), 342 bout-related in all (285 +/- 28.5); reading (a) gives 116, 152, 66 and 334. The pre-bout total, 184, is near the paper's 187; the excess is in post-bout neurons (218 against 149).

## Protosyllables

The Methods define protosyllables by duration: subtract the exponential fitted over 200 to 400 ms from the duration distribution, fit a Gaussian to the residual, keep durations within 2 s.d. of its mean; they are labelled alpha in all birds. In this dataset, birds 1, 2 and 4 carry the label `a` on their protosyllable-stage syllables (the Latin letter for alpha; the Methods' scheme maps alpha, beta, gamma, delta, epsilon to A to E, and the files use lower case for the juvenile letters); for these the labels are used. For the other birds the duration fit is made on all selected syllables of each bird and day of the protosyllable stage. Bird 4's protosyllables were found by phase segmentation in the paper; the dataset's `a` labels are used for it.

In [ ]:
proto_fit = {}
for (b, a), g in J[J.code == 2].groupby(["bird", "age"]):
    d = np.concatenate([np.diff(np.c_[hvc.syllables(NK[k])[1:3]], axis=1).ravel() for k in g.key])
    proto_fit[(b, a)] = hvc.protosyllable_range(d)
def proto_syll(key):
    n = NK[key]
    fi, on, off, lab = hvc.syllables(n)
    if (lab == "a").any():
        k = lab == "a"
    else:
        r = proto_fit.get((n["bird"], int(U.loc[key, "age"])))
        if r is None or not (0.03 < r[2] < 0.3) or r[3] < 0.003:
            return fi[:0], on[:0], off[:0]
        d = off - on
        k = (d >= r[0]) & (d <= r[1])
    return fi[k], on[k], off[k]
print("bird, day: protosyllable duration range from the fit (ms; lo, hi, mean, s.d.), or the label used")
for (b, a), r in proto_fit.items():
    lab_a = any((hvc.syllables(NK[k])[3] == "a").any() for k in J[(J.bird == b) & (J.age == a)].key)
    print(f"  {b} {a}: {'label a' if lab_a else ('fit failed' if r is None else np.round(np.array(r) * 1000, 1).tolist())}")
P2 = J[J.code == 2].copy()
P2["n_proto"] = [proto_syll(k)[1].size for k in P2.key]
print(f"protosyllable-stage neurons with protosyllables: {(P2.n_proto > 0).sum()} of {len(P2)}; protosyllables per neuron {P2.n_proto.min()} to {P2.n_proto.max()}")
# locking to protosyllable onsets
def proto_lock(key):
    fi, on, off = proto_syll(key)
    if on.size == 0:
        return np.nan, np.nan, np.nan
    rel, tri = hvc.aligned(NK[key], on, fi)
    pk, lat, p = hvc.shift_test(rel, tri, on.size, hvc.LOCK_WINDOW, hvc.neuron_rng(key, "protolock"))
    return lat, p, float(np.mean(off - on))
pl = np.array([proto_lock(k) for k in P2.key])
P2["plat"], P2["pp"], P2["pdur"] = pl[:, 0], pl[:, 1], pl[:, 2]
P2["plocked"] = (P2.pp < hvc.ALPHA).astype(int)

## fig2b-protosequence

> In the protosyllable stage, different neurons recorded in the same bird burst at different latencies relative to protosyllable onset, together covering the protosyllable and the following gap (3 birds, 54 neurons; over 90% burst coverage).

In [ ]:
sets = {"to3779": (57, 64, 28), "to3965": (56, 59, 14), "to5640": (48, 49, 12)}
tot = 0
for b, (a0, a1, exp) in sets.items():
    g = P2[(P2.bird == b) & (P2.age.between(a0, a1))]
    L = g[g.plocked == 1]
    tot += len(L)
    print(f"{b} (bird {hvc.BIRD_NUMBER[b]}, {a0}-{a1} dph): {len(g)} neurons, {len(L)} locked to protosyllable onsets (raster has {exp}); "
          f"latencies {L.plat.min()*1000:.0f} to {L.plat.max()*1000:.0f} ms; mean protosyllable {L.pdur.mean()*1000:.0f} ms; "
          f"spans onset to past offset: {bool(L.plat.min() < 0 and L.plat.max() > L.pdur.mean())}")
    print("   sorted latencies (ms):", np.sort(L.plat.values * 1000).round(0).tolist())
print(f"total locked: {tot} (expected 54 +/- 5.4)")

**Result: partially reproduced.** 29, 14 and 12 neurons of birds 2, 1 and 4 are locked to protosyllable onsets, 55 in all (54 +/- 5.4), and the protosyllable coverage is 99.4%, 92.7% and 97.8% (ed2st-coverage). Only in bird 2 do the peak latencies run from before onset (-35 ms) to past the protosyllable's mean offset (188 ms against 132 ms); in birds 1 and 4 the latest peaks, 91 and 95 ms, fall before the mean offsets of 139 and 143 ms.

## ed2p-probabilistic

> Rhythmic protosyllable-stage neurons burst on only about half of the protosyllables (mean 53%).

In [ ]:
for col in ("rhythmic_aa", "rhythmic_bb"):
    R2 = P2[(P2[col] == 1) & (P2.n_proto > 0)]
    fr = np.array([hvc.participation(NK[k], *proto_syll(k)).mean() for k in R2.key])
    print(f"{col}: rhythmic protosyllable-stage neurons {int(P2[col].sum())} (expected 70 +/- 7), with protosyllables {len(R2)}; "
          f"mean fraction of protosyllables with a spike {fr.mean()*100:.1f}% (expected 53 +/- 5.3%), median {np.median(fr)*100:.1f}%, range {fr.min()*100:.0f} to {fr.max()*100:.0f}%")
    lab = np.array([(hvc.syllables(NK[k])[3] == "a").any() for k in R2.key])
    print(f"   labelled birds (1, 2, 4) {fr[lab].mean()*100:.1f}% (n = {lab.sum()}); duration-fit birds {fr[~lab].mean()*100:.1f}% (n = {(~lab).sum()})")

**Result: reproduced.** The 87 rhythmic protosyllable-stage neurons spike on 52.3% of protosyllables on average (53 +/- 5.3%); with 10 ms bins, 88 neurons and 51.6%. The number of rhythmic neurons is above the paper's 70 (fig1f-rhythmic-protosyllable).

## ed2qr-pairs-independent

> Simultaneously recorded rhythmic pairs participate in largely independent sets of protosyllables (median r^2 = 0.072, median mutual information 0.056 bits, 11 pairs).

Simultaneous pairs are neurons of the same bird and day whose analysis files list the same song files. Per protosyllable of the shared files (taken from the first neuron's segments), 1 when the neuron spikes from 30 ms before onset to 10 ms after offset.

In [ ]:
def mi_bits(x, y):
    m = 0.0
    for a in (0, 1):
        for b in (0, 1):
            pab = np.mean((x == a) & (y == b)); pa = np.mean(x == a); pb = np.mean(y == b)
            if pab > 0:
                m += pab * np.log2(pab / (pa * pb))
    return m
pairs = hvc.simultaneous_pairs(N)
print(f"simultaneous pairs in the dataset: {len(pairs)}")
for col in ("rhythmic_aa", "rhythmic_bb"):
    rows = []
    for a, b, nf in pairs:
        if a not in P2.index.values and a not in set(P2.key):
            continue
        ra = P2.set_index("key")
        if a not in ra.index or b not in ra.index or ra.loc[a, col] != 1 or ra.loc[b, col] != 1:
            continue
        na, nb = NK[a], NK[b]
        fi, on, off = proto_syll(a)
        common = [i for i, s in enumerate(na["sound"]) if s in nb["sound"]]
        k = np.isin(fi, common)
        fib = np.array([nb["sound"].index(na["sound"][i]) for i in fi[k]], dtype=int)
        xa = hvc.participation(na, fi[k], on[k], off[k]); xb = hvc.participation(nb, fib, on[k], off[k])
        r = np.corrcoef(xa, xb)[0, 1] if xa.std() > 0 and xb.std() > 0 else np.nan
        rows.append(dict(a=a, b=b, bird=na["bird"], n_proto=int(k.sum()), r2=r ** 2, mi=mi_bits(xa, xb)))
    T = pd.DataFrame(rows)
    print(f"{col}: pairs both rhythmic in the protosyllable stage: {len(T)} in {T.bird.nunique()} birds (expected 11, 3 birds); "
          f"median r^2 = {T.r2.median():.3f} (expected 0.072 +/- 0.0072); median MI = {T.mi.median():.3f} bits (expected 0.056 +/- 0.0056)")
    print(T.round(3).to_string(index=False))

**Result: partially reproduced.** 10 simultaneously recorded pairs are both rhythmic in the protosyllable stage (11 +/- 1.1), from 4 birds where the paper has 3. Their participation is weakly related, but more than in the paper: median r^2 = 0.119 (0.072 +/- 0.0072) and median mutual information 0.083 bits (0.056 +/- 0.0056). With 10 ms bins, 8 pairs, 0.139 and 0.098 bits.

## fig2c-latency-subsong-earlier

> Burst latencies relative to syllable onset are earlier and narrower in subsong than in the protosyllable stage (63% against 43% before onset; P = 0.02).

Neurons locked to syllable onsets (all syllables pooled, both shift readings). Latency readings: (a) one per neuron, the peak of its histogram; (b) all burst times of these neurons within the histogram window (-50 to 200 ms), pooled.

In [ ]:
def lat_b(keys):
    out = []
    for k in keys:
        fi, on, off, lab = hvc.syllables(NK[k])
        out.append(hvc.burst_latencies(NK[k], fi, on))
    return np.concatenate(out)
LAT = {}
for lk, latc in (("locked_b", "lock_lat_wide"), ("locked_a", "lock_lat_circular")):
    A = J[(J.code == 1) & (J[lk] == 1)]; B = J[(J.code == 2) & (J[lk] == 1)]
    la, lb = A[latc].values, B[latc].values
    p = stats.ranksums(la, lb).pvalue
    print(f"shift {lk[-1]}: locked subsong {len(A)} (exp 19 +/- 1.9; {A.bird.nunique()} birds), protosyllable {len(B)} (exp 104 +/- 10.4; {B.bird.nunique()} birds)")
    print(f"  (a) per-neuron peak: before onset {np.mean(la < 0)*100:.0f}% vs {np.mean(lb < 0)*100:.0f}% (exp 63 and 43%); medians {np.median(la)*1000:.1f} vs {np.median(lb)*1000:.1f} ms; rank-sum P = {p:.3f}; IQR {stats.iqr(la)*1000:.0f} vs {stats.iqr(lb)*1000:.0f} ms")
    ba, bb_ = lat_b(A.key), lat_b(B.key)
    print(f"  (b) pooled bursts: n = {len(ba)} vs {len(bb_)}; before onset {np.mean(ba < 0)*100:.0f}% vs {np.mean(bb_ < 0)*100:.0f}%; medians {np.median(ba)*1000:.1f} vs {np.median(bb_)*1000:.1f} ms; rank-sum P = {stats.ranksums(ba, bb_).pvalue:.1e}")
    LAT[lk] = (A, B)

**Result: reproduced (shift reading b, one latency per neuron).** 64% of locked subsong neurons and 44% of locked protosyllable-stage neurons have their peak before onset (63% and 43% expected), the subsong median is earlier (-14.5 against 6.5 ms) and the rank-sum P is 0.038. Under shift reading (a) the percentages are 59% and 44% with P = 0.143, and with pooled burst times the fractions are 29% and 23%. The numbers of locked neurons, 25 and 115, are above the paper's 19 and 104.

## fig2c-latency-broadens-later

> Over development, burst latencies spread more uniformly through the syllable, with the multi-syllable and motif stages broader than the protosyllable stage.

In [ ]:
for lk, latc in (("locked_b", "lock_lat_wide"), ("locked_a", "lock_lat_circular")):
    B = J[(J.code == 2) & (J[lk] == 1)]; C = J[(J.code >= 3) & (J[lk] == 1)]
    lb, lc = B[latc].values, C[latc].values
    print(f"shift {lk[-1]}: locked multi-syllable/motif neurons {len(C)} (expected 814 +/- 81.4)")
    print(f"  (a) IQR protosyllable {stats.iqr(lb)*1000:.1f} ms vs multi/motif {stats.iqr(lc)*1000:.1f} ms; before onset {np.mean(lb<0)*100:.0f}% vs {np.mean(lc<0)*100:.0f}%; "
          f"s.d. {lb.std()*1000:.1f} vs {lc.std()*1000:.1f} ms; rank-sum P = {stats.ranksums(lb, lc).pvalue:.3f}")
    bb_, bc = lat_b(B.key), lat_b(C.key)
    print(f"  (b) pooled bursts IQR {stats.iqr(bb_)*1000:.1f} vs {stats.iqr(bc)*1000:.1f} ms; before onset {np.mean(bb_<0)*100:.0f}% vs {np.mean(bc<0)*100:.0f}%; rank-sum P = {stats.ranksums(bb_, bc).pvalue:.1e}")

**Result: reproduced.** With one latency per neuron, the 851 locked multi-syllable and motif neurons (814 +/- 81.4) have an IQR of latencies of 101.0 ms against 52.0 ms in the protosyllable stage, and 36% before onset against 44% (rank-sum P = 0.002). Pooled burst times (reading b) show no broadening (IQR 130.5 against 137.4 ms).

## Shared and specific neurons: the syllable pairs

Per neuron and syllable type, the time-warped, onset-aligned histogram and its shift test (`hvc.type_test`; Methods, Shared and specific neurons). The Methods say the interval was "adjusted if necessary to avoid the same burst being detected twice (that is, being associated with an offset of one syllable and an onset of the next syllable)", by visual inspection. That cannot be repeated; the main reading here replaces it by a rule (an assumption): when a type's peak falls in the 30 ms before onset or the 10 ms after offset, the peak is searched again, and tested, within the syllable itself. The unadjusted result (suffix `_raw`) is shown as the other reading. Without the rule, bursts in the gap between beta and gamma are counted for both syllables, at latencies about one syllable apart (shown under ed4a-shared-latency-bird1). With linear time-warping, offset alignment gives the same histogram shifted by the mean duration, so onset- and offset-aligned classes are the same (shown for bird 2 below). The dataset's labels are Latin letters: lower case `a` to `e` for the juvenile alpha to epsilon and capitals for the adult syllables (Methods, Syllable classification and labelling). The pairs and periods:

- bird 1 (to3965) beta/gamma: `b` or `B` against `c` or `C`, early 62 to 72 dph (the labels switch from `b`, `c` to `B`, `C` at 70 dph), late 81 to 112 dph `B` against `C` (Fig. 3h); variant with `D` added to beta at 70 to 72 dph (`D` is the first beta of a bout, Methods);
- bird 1 B/D (Extended Data Fig. 7): `B` against `D`; the early/late boundary is not given; readings (a) 80 dph, when D and B begin to differ (Extended Data Fig. 7 legend, "~80 dph, panel c"), (b) 85 dph;
- bird 2 (to3779) alpha/beta: `a` against `b`, early 65 to 72, late 73 to 84 dph (Fig. 4);
- bird 4 (to5640) b/d and c/d: the boundary is not given; readings (a) the list's stage boundary (multi-syllable 51 to 58, motif 59 to 67 dph), (b) 51 to 55 against 60 to 67 dph;
- bird 5 (to3396), bout-onset differentiation, data not shown in the paper: `a` against `b`, read as in bird 2 (assumption);
- bird 6 (to2309), hierarchical differentiation, data not shown: the pair is not given; every pair of its labelled types is run and shown;
- bird 3 (to3017): only one of its 19 neurons has syllable labels (148 syllables), so no pair can be formed.

In [ ]:
def keys_of(bird, a0, a1):
    return [k for k in J.key if NK[k]["bird"] == bird and a0 <= U.loc[k, "age"] <= a1]
def ptab(bird, a0, a1, A, B, align="onset", reading="wide", adjust=True):
    T = hvc.pair_table([NK[k] for k in keys_of(bird, a0, a1)], A, B, align, reading, adjust=adjust)
    T["tab"] = T.key.map(U.tab); T["age"] = T.key.map(U.age)
    return T
def counts(T):
    c = T.cls.value_counts()
    return dict(n=len(T), active=int((T.cls != "none").sum()), shared=int(c.get("shared", 0)),
                spec_A=int(c.get("specific_A", 0)), spec_B=int(c.get("specific_B", 0)))
t = time.time()
PT = {}
PT["b1_bg_early"] = ptab("to3965", 62, 72, {"b", "B"}, {"c", "C"})
PT["b1_bg_early_D"] = ptab("to3965", 62, 72, {"b", "B", "D"}, {"c", "C"})
PT["b1_bg_early_circ"] = ptab("to3965", 62, 72, {"b", "B"}, {"c", "C"}, reading="circular")
PT["b1_BC_late"] = ptab("to3965", 81, 112, {"B"}, {"C"})
PT["b1_BC_late_circ"] = ptab("to3965", 81, 112, {"B"}, {"C"}, reading="circular")
PT["b1_BD_early_a"] = ptab("to3965", 70, 79, {"B"}, {"D"}); PT["b1_BD_late_a"] = ptab("to3965", 80, 112, {"B"}, {"D"})
PT["b1_BD_early_b"] = ptab("to3965", 70, 84, {"B"}, {"D"}); PT["b1_BD_late_b"] = ptab("to3965", 85, 112, {"B"}, {"D"})
PT["b2_ab_early"] = ptab("to3779", 65, 72, {"a"}, {"b"}); PT["b2_ab_late"] = ptab("to3779", 73, 84, {"a"}, {"b"})
PT["b2_ab_early_off"] = ptab("to3779", 65, 72, {"a"}, {"b"}, align="offset"); PT["b2_ab_late_off"] = ptab("to3779", 73, 84, {"a"}, {"b"}, align="offset")
PT["b2_ab_early_circ"] = ptab("to3779", 65, 72, {"a"}, {"b"}, reading="circular"); PT["b2_ab_late_circ"] = ptab("to3779", 73, 84, {"a"}, {"b"}, reading="circular")
for nm, (e0, e1, l0, l1) in {"a": (51, 58, 59, 67), "b": (51, 55, 60, 67)}.items():
    PT[f"b4_bd_early_{nm}"] = ptab("to5640", e0, e1, {"b"}, {"d"}); PT[f"b4_bd_late_{nm}"] = ptab("to5640", l0, l1, {"b"}, {"d"})
    PT[f"b4_cd_early_{nm}"] = ptab("to5640", e0, e1, {"c"}, {"d"}); PT[f"b4_cd_late_{nm}"] = ptab("to5640", l0, l1, {"c"}, {"d"})
PT["b5_ab"] = ptab("to3396", 0, 200, {"a"}, {"b"})
# the same pairs without the window adjustment (variant "raw")
for k_, args in {"b1_bg_early": ("to3965", 62, 72, {"b", "B"}, {"c", "C"}), "b1_BC_late": ("to3965", 81, 112, {"B"}, {"C"}),
                 "b1_BD_early_a": ("to3965", 70, 79, {"B"}, {"D"}), "b1_BD_late_a": ("to3965", 80, 112, {"B"}, {"D"}),
                 "b2_ab_early": ("to3779", 65, 72, {"a"}, {"b"}), "b2_ab_late": ("to3779", 73, 84, {"a"}, {"b"}),
                 "b4_bd_early_a": ("to5640", 51, 58, {"b"}, {"d"}), "b4_bd_late_a": ("to5640", 59, 67, {"b"}, {"d"}),
                 "b4_cd_early_a": ("to5640", 51, 58, {"c"}, {"d"}), "b4_cd_late_a": ("to5640", 59, 67, {"c"}, {"d"}),
                 "b5_ab": ("to3396", 0, 200, {"a"}, {"b"})}.items():
    PT[k_ + "_raw"] = ptab(*args, adjust=False)
b6_types = sorted({t for k in keys_of("to2309", 0, 200) for t in hvc.syllables(NK[k])[3] if t})
for i, x in enumerate(b6_types):
    for y in b6_types[i + 1:]:
        PT[f"b6_{x}{y}"] = ptab("to2309", 0, 200, {x}, {y})
print(f"pair tables in {time.time()-t:.0f} s")
CT = pd.DataFrame({k: counts(v) for k, v in PT.items()}).T
print(CT.to_string())

## fig3g-bird1-shared-specific

> After syllables beta and gamma emerge in bird 1 (62 to 72 dph), of 105 neurons active in either, 41 are beta-specific, 42 gamma-specific and 22 shared.

In [ ]:
for k in ("b1_bg_early", "b1_bg_early_raw", "b1_bg_early_circ", "b1_bg_early_D"):
    c = counts(PT[k])
    print(f"{k}: neurons {c['n']}, active in beta or gamma {c['active']} (exp 105 +/- 10.5), beta-specific {c['spec_A']} (exp 41 +/- 4.1), "
          f"gamma-specific {c['spec_B']} (exp 42 +/- 4.2), shared {c['shared']} (exp 22 +/- 2.2)")
T = PT["b1_bg_early"]
print("by age, early bird 1:"); print(pd.crosstab(T.age, T.cls).to_string())
sh_ = T[T.cls == "shared"]
print("shared neurons, P values for beta and gamma:", list(zip(sh_.p_A.round(3), sh_.p_B.round(3))))
for thr in (0.05, 0.01, 0.001):
    sa, sb = T.p_A < thr, T.p_B < thr
    print(f"  threshold {thr}: shared {int((sa & sb).sum())}, beta-specific {int((sa & ~sb).sum())}, gamma-specific {int((sb & ~sa).sum())}")

**Result: reproduced, resting on the window adjustment (an assumption).** With the adjusted window, 96 neurons of bird 1 at 62 to 72 dph are active in beta or gamma (105 +/- 10.5): 37 beta-specific (41 +/- 4.1), 39 gamma-specific (42 +/- 4.2) and 20 shared (22 +/- 2.2). Without the adjustment there are 32 shared neurons, outside the tolerance, as bursts in the gap between the syllables count for both.

## fig3h-bird1-shared-decrease

> In bird 1 the fraction of shared neurons is lower late in development (81 to 112 dph; 10 shared and 90 specific) than early (P = 0.03).

In [ ]:
def chi2_2x2(c1, c2, label):
    tab = np.array([[c1["shared"], c1["spec_A"] + c1["spec_B"]], [c2["shared"], c2["spec_A"] + c2["spec_B"]]])
    exp_ = stats.contingency.expected_freq(tab)
    if (exp_ < 5).any():
        p = stats.fisher_exact(tab)[1]; test = "Fisher"
    else:
        p = stats.chi2_contingency(tab, correction=False)[1]; test = "chi-square"
    pc = stats.chi2_contingency(tab, correction=True)[1]
    f1, f2 = tab[0, 0] / tab[0].sum(), tab[1, 0] / tab[1].sum()
    print(f"{label}: early {tab[0].tolist()} ({f1:.2f} shared), late {tab[1].tolist()} ({f2:.2f} shared); {test} P = {p:.2g} (Yates {pc:.2g})")
    return tab, p
for e, l in (("b1_bg_early", "b1_BC_late"), ("b1_bg_early_raw", "b1_BC_late_raw"), ("b1_bg_early_circ", "b1_BC_late_circ")):
    c2 = counts(PT[l])
    print(f"late {l}: shared {c2['shared']} (exp 10 +/- 1), specific {c2['spec_A'] + c2['spec_B']} (exp 90 +/- 9)")
    chi2_2x2(counts(PT[e]), c2, f"{e} vs {l}")
print("paper's counts (22, 83) vs (10, 90): P = %.3f" % stats.chi2_contingency(np.array([[22, 83], [10, 90]]), correction=False)[1])

**Result: partially reproduced.** The shared fraction falls from 20 of 96 early to 8 of 92 late (chi-square P = 0.019), with 84 late specific neurons (90 +/- 9), but the 8 late shared neurons are outside 10 +/- 1. Without the window adjustment, 12 late shared and 92 specific (P = 0.001).

## ed4a-shared-latency-bird1

> In bird 1, each shared neuron bursts at a similar latency in beta and in gamma (r = 0.91 early, 0.87 late).

In [ ]:
def lat_corr(T, label, drop=None):
    s = T[T.cls == "shared"]
    if drop is not None:
        s = s[~drop(s)]
    if len(s) < 3:
        print(f"{label}: {len(s)} shared neurons, no correlation"); return np.nan, np.nan, len(s)
    r, p = stats.pearsonr(s.lat_A, s.lat_B)
    d = (s.lat_A - s.lat_B).abs() * 1000
    print(f"{label}: n = {len(s)}, r = {r:.3f}, P = {p:.1e}; |latency difference| median {d.median():.1f} ms, max {d.max():.1f} ms")
    return r, p, len(s)
for k in ("b1_bg_early", "b1_BC_late", "b1_bg_early_raw", "b1_BC_late_raw", "b1_bg_early_circ", "b1_BC_late_circ"):
    lat_corr(PT[k], k)
s_ = PT["b1_bg_early_raw"]; s_ = s_[s_.cls == "shared"]
far = s_[(s_.lat_A - s_.lat_B).abs() > 0.1]
print(f"raw reading: shared neurons with latencies more than 100 ms apart: {len(far)}; their latencies (ms, beta, gamma):",
      list(zip((far.lat_A * 1000).round(0), (far.lat_B * 1000).round(0))), "; mean beta duration", round(s_.mean_dur_A.mean() * 1000), "ms")
for k_ in ("b1_bg_early", "b1_BC_late"):
    s = PT[k_][PT[k_].cls == "shared"]
    A_, B_ = ({"b", "B"}, {"c", "C"}) if "early" in k_ else ({"B"}, {"C"})
    la = np.array([hvc.peak_latency(NK[k], A_, "onset", warp=False) for k in s.key]); lb = np.array([hvc.peak_latency(NK[k], B_, "onset", warp=False) for k in s.key])
    r, p = stats.pearsonr(la, lb)
    print(f"{k_}, onset-aligned without warping (variant): n = {len(s)}, r = {r:.3f}, P = {p:.1e}")
print("expected r = 0.91 +/- 0.091 (early), 0.87 +/- 0.087 (late)")

**Result: partially reproduced, resting on the window adjustment.** Early, the 20 shared neurons' latencies in beta and gamma correlate with r = 0.852 (P = 1.8e-06; 0.91 +/- 0.091 expected). Late, the 8 shared neurons give r = 0.700 (P = 0.053), below 0.87 +/- 0.087; without warping, 0.713 (P = 0.047). Without the adjustment both correlations vanish (r = -0.210 and 0.059), because 11 early shared neurons have one peak at the end of beta and the other just before gamma.

## ed3j-shared-ibi-specific-doubled

> In bird 1, shared neurons keep the inter-burst interval of protosyllable-stage neurons, while syllable-specific neurons have an interval twice as long.

Most-probable IBI per neuron: the peak of its IBI distribution between 80 and 1,000 ms, under bin readings (a) and (b). The protosyllable group is shown for all 16 bird-1 protosyllable-stage neurons and for its rhythmic neurons.

In [ ]:
Ji = J.set_index("key")
T = PT["b1_bg_early"]
for k_ in ("b1_bg_early", "b1_bg_early_raw"):
    print(k_, "shared", int((PT[k_].cls == "shared").sum()), "specific", int(PT[k_].cls.str.startswith("specific").sum()))
g_sh = T[T.cls == "shared"].key; g_sp = T[T.cls.str.startswith("specific")].key
g_pr = J[(J.bird == "to3965") & (J.code == 2)].key
g_pr_r = J[(J.bird == "to3965") & (J.code == 2) & (J.rhythmic_aa == 1)].key
for rd in "ab":
    m = {nm: Ji.loc[g, f"ibi_lag_{rd}"] * 1000 for nm, g in (("proto", g_pr), ("proto_rhythmic", g_pr_r), ("shared", g_sh), ("specific", g_sp))}
    for base in ("proto", "proto_rhythmic"):
        print(f"bins {rd}, base {base}: n {len(m[base])}/{len(m['shared'])}/{len(m['specific'])}; medians proto {m[base].median():.0f}, shared {m['shared'].median():.0f}, specific {m['specific'].median():.0f} ms; "
              f"ratio specific/proto {m['specific'].median()/m[base].median():.2f} (exp 2 +/- 0.2), shared/proto {m['shared'].median()/m[base].median():.2f} (exp 1 +/- 0.1); "
              f"rank-sum shared vs proto P = {stats.ranksums(m['shared'], m[base]).pvalue:.2g}, specific vs proto P = {stats.ranksums(m['specific'], m[base]).pvalue:.2g}")
print("expected protosyllable-stage n = 9 +/- 0.9")

**Result: reproduced (10 ms bins).** The median most-probable IBI is 180 ms for bird 1's protosyllable-stage neurons, 185 ms for shared and 395 ms for specific neurons: ratios 2.19 (2 +/- 0.2) and 1.03 (1 +/- 0.1). With 1 ms bins the specific ratio is 2.25, just outside. The protosyllable group has 16 neurons (12 rhythmic), where the paper has 9.

## all-birds-shared-specific

> Shared and specific neurons are found in all 6 birds recorded during the emergence of several syllable types (185 shared and 496 specific neurons over 8 syllable pairs).

Pairs as listed above; bird 3 has no labels, and bird 6's pair is not known, so every bird-6 pair is shown and the totals are given with the bird-6 pair that has the fewest and the most shared neurons. Readings: (a) all ages of each pair, (b) the early period only.

In [ ]:
def tot(keys):
    c = [counts(PT[k]) for k in keys]
    return sum(x["shared"] for x in c), sum(x["spec_A"] + x["spec_B"] for x in c)
b6 = [k for k in PT if k.startswith("b6_")]
b6s = sorted(b6, key=lambda k: counts(PT[k])["shared"])
print("bird 6 pairs (shared, specific):", {k: (counts(PT[k])["shared"], counts(PT[k])["spec_A"] + counts(PT[k])["spec_B"]) for k in b6})
read_a = ["b1_bg_early", "b1_BC_late", "b1_BD_early_a", "b1_BD_late_a", "b2_ab_early", "b2_ab_late", "b4_bd_early_a", "b4_bd_late_a", "b4_cd_early_a", "b4_cd_late_a", "b5_ab"]
read_b = ["b1_bg_early", "b1_BD_early_a", "b2_ab_early", "b4_bd_early_a", "b4_cd_early_a", "b5_ab"]
for lab, ks in (("(a) all ages", read_a), ("(b) early only", read_b), ("(a) all ages, raw", [k + "_raw" for k in read_a]), ("(b) early only, raw", [k + "_raw" for k in read_b])):
    for b6k in (b6s[0], b6s[-1]):
        s_, p_ = tot(ks + [b6k])
        print(f"{lab}, bird 6 pair {b6k}: shared {s_} (exp 185 +/- 18.5), specific {p_} (exp 496 +/- 49.6); pairs counted {len(set(k.rsplit('_', 1)[0] if k.endswith(('early','late','early_a','late_a')) else k for k in ks)) + 1}")
print("\nper bird, does the bird have at least one shared and one specific neuron (any pair)?")
for b, pref in (("bird 1", "b1_"), ("bird 2", "b2_"), ("bird 3", "b3_"), ("bird 4", "b4_"), ("bird 5", "b5_"), ("bird 6", "b6_")):
    ks = [k for k in PT if k.startswith(pref)]
    s_ = sum(counts(PT[k])["shared"] for k in ks); p_ = sum(counts(PT[k])["spec_A"] + counts(PT[k])["spec_B"] for k in ks)
    print(f"  {b}: {'no labelled pair' if not ks else f'shared {s_ > 0}, specific {p_ > 0}'}")

**Result: partially reproduced, resting on assumptions about the pairs.** Birds 1, 2, 4, 5 and 6 each have shared and specific neurons; bird 3 cannot be tested, as its syllables carry no labels. Over all ages (reading a) the adjusted counts give 167 or 168 shared (185 +/- 18.5) and 428 specific (496 +/- 49.6), and the unadjusted 233 or 234 shared and 468 specific; under no reading are both totals inside. Early periods only (reading b) give 105 and 205.

## all-birds-shared-decrease

> Across the three birds also recorded later in development (birds 1, 2 and 4; 5 syllable pairs), the fraction of shared neurons decreases during syllable development (P = 3 x 10^-6).

In [ ]:
from statsmodels.stats.contingency_tables import StratifiedTable
for rd in ("a", "b", "raw"):
    pairs5 = [("b1_bg_early", "b1_BC_late"), (f"b1_BD_early_{rd}", f"b1_BD_late_{rd}"), ("b2_ab_early", "b2_ab_late"),
              (f"b4_bd_early_{rd}", f"b4_bd_late_{rd}"), (f"b4_cd_early_{rd}", f"b4_cd_late_{rd}")]
    if rd == "raw":   # boundary reading (a), without the window adjustment
        pairs5 = [(e + "_raw", l + "_raw") for e, l in [("b1_bg_early", "b1_BC_late"), ("b1_BD_early_a", "b1_BD_late_a"), ("b2_ab_early", "b2_ab_late"),
                                                         ("b4_bd_early_a", "b4_bd_late_a"), ("b4_cd_early_a", "b4_cd_late_a")]]
    tabs = []
    for e, l in pairs5:
        ce, cl = counts(PT[e]), counts(PT[l])
        tab = np.array([[ce["shared"], ce["spec_A"] + ce["spec_B"]], [cl["shared"], cl["spec_A"] + cl["spec_B"]]])
        tabs.append(tab)
        print(f"  boundary reading {rd}, {e} / {l}: early {tab[0].tolist()}, late {tab[1].tolist()}")
    st = StratifiedTable([t.astype(float) for t in tabs])
    pooled = np.sum(tabs, axis=0)
    for corr in (False, True):
        r = st.test_null_odds(correction=corr)
        print(f"boundary {rd}, CMH {'with' if corr else 'without'} continuity correction: statistic {r.statistic:.2f}, P = {r.pvalue:.2g}; "
              f"pooled shared fraction early {pooled[0,0]/pooled[0].sum():.3f}, late {pooled[1,0]/pooled[1].sum():.3f}; pooled odds ratio {st.oddsratio_pooled:.2f}")
print("expected P = 3e-6")

**Result: reproduced, resting on assumptions about the boundaries.** Over the five pairs of birds 1, 2 and 4, the pooled shared fraction falls from 0.325 to 0.218 (boundary reading a; CMH P = 1.3e-04) or from 0.406 to 0.184 (reading b; P = 1.7e-06), and without the window adjustment P = 1.2e-07; the paper's P is 3e-6.

## skipping-cycles

> Neurons whose burst period is lengthened by skipping cycles of the underlying rhythm are found in 4 of the 6 birds (birds 1, 3, 4 and 6).

Base period P0 per bird: the median period (autocorrelation peak) of its rhythmic protosyllable-stage neurons, or, failing those, of its shared neurons in the pairs above. Reading (a): a rhythmic neuron of a later stage with period within 20% of 2 x P0; reading (b): a rhythmic neuron whose IBI distribution (bins a) has its largest peak within 20% of 2 x P0 and a smaller local peak within 20% of P0.

In [ ]:
def ibi_hist(key):
    bb = hvc._bout_bursts(NK[key])
    real = [np.sort(np.concatenate(b))[None, :] if b else np.zeros((1, 0)) for T_, b in bb]
    H = hvc._lag_hist(real, "ibi", 1)
    return (hvc._smooth_rows(H) / hvc.SMOOTH)[0]
def has_peak_near(h, lag, tol=0.2):
    lo, hi = int(lag * (1 - tol) * 1000), int(lag * (1 + tol) * 1000)
    seg = h[max(lo, 1):min(hi, 999)]
    if seg.size < 3:
        return False
    i = np.argmax(seg) + max(lo, 1)
    return bool(h[i] > 0 and h[i] >= h[i - 1] and h[i] >= h[i + 1] and lo < i < hi - 1)
for b in ("to3965", "to3779", "to3017", "to5640", "to3396", "to2309"):
    G = J[J.bird == b]
    pr = G[(G.code == 2) & (G.rhythmic_aa == 1)]
    if len(pr):
        P0 = pr.period_a.median(); src = f"{len(pr)} rhythmic protosyllable-stage neurons"
    else:
        shk = [k for pk, T in PT.items() if pk.startswith({"to3396": "b5_", "to2309": "b6_"}.get(b, "zz")) for k in T[T.cls == "shared"].key]
        shk = [k for k in dict.fromkeys(shk) if Ji.loc[k, "rhythmic_aa"] == 1]
        if not shk:
            print(f"{b} (bird {hvc.BIRD_NUMBER[b]}): no base period"); continue
        P0 = Ji.loc[shk, "period_a"].median(); src = f"{len(shk)} rhythmic shared neurons"
    later = G[(G.code >= 2) & (G.rhythmic_aa == 1)] if len(pr) == 0 else G[(G.code >= 3) & (G.rhythmic_aa == 1)]
    if b == "to5640" or b == "to3017":
        later = G[G.rhythmic_aa == 1]   # birds 3 and 4: the paper's examples include protosyllable-stage neurons (Extended Data Figs 8, 9f)
    a_ = later[(later.period_a - 2 * P0).abs() <= 0.2 * 2 * P0]
    bk = []
    for k in later.key:
        h = ibi_hist(k)
        lo, hi = int(2 * P0 * 0.8 * 1000), int(2 * P0 * 1.2 * 1000)
        m = (np.arange(1000) >= 80)
        imax = np.argmax(np.where(m, h, -1))
        if lo <= imax <= hi and has_peak_near(h, P0):
            bk.append(k)
    print(f"{b} (bird {hvc.BIRD_NUMBER[b]}): P0 = {P0*1000:.0f} ms from {src}; rhythmic neurons tested {len(later)}; (a) {len(a_)} at about 2 x P0; (b) {len(bk)} with IBI peaks at 2 x P0 and P0")

**Result: partially reproduced.** Under reading (a), rhythmic neurons with a period near twice the bird's protosyllable period are found in bird 1 (242), bird 3 (1) and bird 4 (14), but not in bird 6, whose base period rests on a single rhythmic shared neuron (631 ms); birds 2 and 5 have none, as in the paper. Reading (b) finds 99, 1 and 4 in the same three birds.

## fig4-bird2-before-beta

> In bird 2 (125 neurons), before syllable beta emerges (57 to 64 dph), most neurons (28 of 35) take part in a rhythmic protosequence and a different 4 of 35 burst at bout onsets.

In [ ]:
print(f"bird 2 neurons: files {sum(1 for n in N if n['bird']=='to3779')}, kept {int((J.bird=='to3779').sum())} (expected 125 +/- 12.5)")
G = P2[(P2.bird == "to3779") & P2.age.between(57, 64)].set_index("key")
print(f"57-64 dph: {len(G)} (expected 35 +/- 3.5)")
pb = Ji.loc[G.index, "pre_bout"]
for lab, m in (("(a) locked to protosyllable onsets", G.plocked == 1), ("(b) rhythmic", G.rhythmic_aa == 1), ("(c) both", (G.plocked == 1) & (G.rhythmic_aa == 1))):
    print(f"{lab}: {int(m.sum())} (expected 28 +/- 2.8); pre-bout neurons {int(pb.sum())} (expected 4), of which outside the protosequence {int((pb.astype(bool) & ~m).sum())}")
print("pre-bout P (shift b) of the pre-bout neurons:", Ji.loc[pb[pb == 1].index, "pre_p_wide"].round(3).tolist())

**Result: partially reproduced.** Bird 2 has 125 neurons and 35 at 57 to 64 dph, as in the paper; 29 are locked to protosyllable onsets (reading a) and 28 are rhythmic (reading b), against 28 +/- 2.8. By the pre-bout test of fig1de-bout-related, 12 of the 35 burst before bout onset (4 expected); 4 of these 12 are outside the rhythmic group.

## bird1-protosequence

> In bird 1's protosyllable stage (56 to 59 dph), most projection neurons (14 of 16) take part in a rhythmic protosequence.

In [ ]:
G = P2[(P2.bird == "to3965") & P2.age.between(56, 59)]
print(f"bird 1 protosyllable-stage neurons: {len(G)} (expected 16 +/- 1.6)")
for lab, m in (("(a) locked to protosyllable onsets", G.plocked == 1), ("(b) rhythmic", G.rhythmic_aa == 1), ("(b') rhythmic, bins b", G.rhythmic_bb == 1), ("(c) both", (G.plocked == 1) & (G.rhythmic_aa == 1))):
    print(f"{lab}: {int(m.sum())} (expected 14 +/- 1.4)")

**Result: reproduced.** Bird 1 has 16 protosyllable-stage neurons (56 to 59 dph), and 14 are locked to protosyllable onsets (reading a). 12 are rhythmic (reading b) and 11 are both (reading c).

## fig4h-bird2-shared-early

> After syllable beta emerges at bout onset in bird 2 (65 to 72 dph), about half the neurons are shared between alpha and beta (22 shared, 21 specific), and shared neurons produce nearly identical sequences in the two syllables (r > 0.99).

Offset-aligned as Fig. 4h (identical to onset-aligned under linear warping; both shown). The shared neuron with two peaks in alpha is identified as the shared neuron whose alpha histogram has two separate peaks above half its maximum in the window, at least 30 ms apart (an operational choice for the legend's "had two peaks").

In [ ]:
for k in ("b2_ab_early_off", "b2_ab_early", "b2_ab_early_raw", "b2_ab_early_circ"):
    c = counts(PT[k])
    print(f"{k}: neurons {c['n']}, shared {c['shared']} (exp 22 +/- 2.2), specific {c['spec_A'] + c['spec_B']} (exp 21 +/- 2.1; alpha {c['spec_A']}, beta {c['spec_B']})")
def two_peaks(key, labels, md):
    n = NK[key]; fi, on, off, _ = hvc.syllables(n, labels)
    rel, tri = hvc.aligned(n, on, fi, warp=md / (off - on))
    h, c = hvc.histogram(rel, on.size)
    w = (c >= -hvc.PRE_ONSET) & (c <= md + hvc.POST_OFFSET)
    x = h[w]; thr = 0.5 * x.max()
    pk = [i for i in range(1, len(x) - 1) if x[i] >= thr and x[i] >= x[i - 1] and x[i] > x[i + 1]]
    groups = [pk[0]] if pk else []
    for i in pk[1:]:
        if i - groups[-1] >= 30:
            groups.append(i)
    return len(groups) >= 2
T = PT["b2_ab_early"]
s = T[T.cls == "shared"]
dbl = s.apply(lambda r: two_peaks(r.key, {"a"}, r.mean_dur_A), axis=1)
print(f"shared neurons with two peaks in alpha: {int(dbl.sum())}")
lat_corr(T, "early, all shared")
lat_corr(PT["b2_ab_early_raw"], "early, raw reading, all shared")
r_e = lat_corr(T, "early, two-peak neurons left out", drop=lambda s: s.apply(lambda r: two_peaks(r.key, {"a"}, r.mean_dur_A), axis=1))
# variant: latencies from offset-aligned histograms without time-warping (beta = epsilon + alpha, so its onset is earlier by epsilon)
for k_ in ("b2_ab_early", "b2_ab_early_raw"):
    s = PT[k_][PT[k_].cls == "shared"]
    la = np.array([hvc.peak_latency(NK[k], {"a"}, "offset", warp=False) for k in s.key]); lb = np.array([hvc.peak_latency(NK[k], {"b"}, "offset", warp=False) for k in s.key])
    r, p = stats.pearsonr(la, lb)
    print(f"{k_}, offset-aligned without warping: n = {len(s)}, r = {r:.3f}, P = {p:.1e}; |latency difference| median {np.median(np.abs(la - lb))*1000:.1f} ms")
print("expected r above 0.891 (bound 0.99 widened by 10%), P < 0.05")

**Result: partially reproduced.** With the adjusted window, bird 2 at 65 to 72 dph has 20 shared (22 +/- 2.2) and 22 specific (21 +/- 2.1; 23 offset-aligned) neurons, but the shared neurons' latencies in alpha and beta correlate with r = 0.787 (P = 3.9e-05), below 0.891; no shared neuron has two peaks in alpha by the rule used. Unadjusted, 25 shared and r = -0.144.

## fig4i-bird2-shared-decrease

> Later in bird 2's development (73 to 84 dph) shared neurons are fewer (4 shared, 28 specific; P = 5 x 10^-4).

In [ ]:
for e, l in (("b2_ab_early_off", "b2_ab_late_off"), ("b2_ab_early_raw", "b2_ab_late_raw"), ("b2_ab_early_circ", "b2_ab_late_circ")):
    cl = counts(PT[l])
    print(f"{l}: shared {cl['shared']} (exp 4 +/- 0.4), specific {cl['spec_A'] + cl['spec_B']} (exp 28 +/- 2.8)")
    chi2_2x2(counts(PT[e]), cl, f"{e} vs {l}")
print("paper's counts (22, 21) vs (4, 28): P = %.1e" % stats.chi2_contingency(np.array([[22, 21], [4, 28]]), correction=False)[1])

**Result: partially reproduced.** The shared fraction falls from 20 of 43 early to 6 of 31 late (chi-square P = 0.016), but the late counts, 6 shared and 25 specific, are outside 4 and 28 +/- 2.8. Unadjusted, 8 shared and 25 specific.

## ed4bd-shared-latency-other

> Shared neurons burst at similar latencies in both syllables of the other pairs as well; bird 1 D/B and bird 2 alpha/beta both stages r > 0.99, bird 4 b2/d1 r = 0.89 and c/d2 r = 0.98.

Bird 4's halves: b1/b2 and d1/d2 are the first and second halves of b and d (Extended Data Fig. 9g, h). For b2/d1, shared b/d neurons whose b latency is in the first half of b (b1) are left out (legend) and the b latency is taken from the middle of b; for c/d2, neurons whose d latency is in the second half of d are kept and the d latency is taken from the middle of d.

In [ ]:
for k in ("b1_BD_early_a", "b1_BD_late_a", "b1_BD_early_b", "b1_BD_late_b", "b2_ab_early", "b2_ab_late", "b1_BD_early_a_raw", "b1_BD_late_a_raw", "b2_ab_early_raw", "b2_ab_late_raw"):
    lat_corr(PT[k], k)
for k_ in ("b2_ab_early", "b2_ab_late"):
    s = PT[k_][PT[k_].cls == "shared"]
    la = np.array([hvc.peak_latency(NK[k], {"a"}, "offset", warp=False) for k in s.key]); lb = np.array([hvc.peak_latency(NK[k], {"b"}, "offset", warp=False) for k in s.key])
    r, p = stats.pearsonr(la, lb) if len(s) > 2 else (np.nan, np.nan)
    print(f"{k_}, offset-aligned without warping (variant): n = {len(s)}, r = {r:.3f}, P = {p:.1e}")
for rd in ("a", "b"):
    T = PT[f"b4_bd_early_{rd}"].copy()
    T2 = T[(T.cls == "shared") & (T.lat_A >= T.mean_dur_A / 2)].copy()
    T2["lat_A"] = T2.lat_A - T2.mean_dur_A / 2
    print(f"bird 4 b/d early ({rd}): shared {int((T.cls=='shared').sum())}, with b latency in b2 {len(T2)}")
    lat_corr(T2.assign(cls="shared"), f"  b2/d1 ({rd})")
    T = PT[f"b4_cd_early_{rd}"].copy()
    T3 = T[(T.cls == "shared") & (T.lat_B >= T.mean_dur_B / 2)].copy()
    T3["lat_B"] = T3.lat_B - T3.mean_dur_B / 2
    print(f"bird 4 c/d early ({rd}): shared {int((T.cls=='shared').sum())}, with d latency in d2 {len(T3)}")
    lat_corr(T3.assign(cls="shared"), f"  c/d2 ({rd})")
print("expected: bird 1 D/B and bird 2 r above 0.891; bird 4 b2/d1 0.89 +/- 0.089, c/d2 0.98 +/- 0.098; all P < 0.05")

**Result: not reproduced.** Bird 1's D/B correlations meet the bound (early r = 0.998, late 0.991, n = 38 and 40), and bird 4's c/d2 gives 0.957 (P = 0.003; 0.98 +/- 0.098). Bird 4's b2/d1 gives 0.757 (P = 0.030), below 0.89 +/- 0.089, and bird 2's alpha/beta gives 0.787 early and -0.406 late (n = 6, P = 0.42), so not every correlation is positive and significant.

## ed7-bird1-bd-shared-persist

> In bird 1 a large fraction of neurons shared between B and D remains in the later recordings, larger than the fraction shared between B and C at late stages.

In [ ]:
cBC = counts(PT["b1_BC_late"])
fBC = cBC["shared"] / (cBC["shared"] + cBC["spec_A"] + cBC["spec_B"])
for rd in ("a", "b"):
    ce, cl = counts(PT[f"b1_BD_early_{rd}"]), counts(PT[f"b1_BD_late_{rd}"])
    fl = cl["shared"] / max(cl["shared"] + cl["spec_A"] + cl["spec_B"], 1)
    tab = np.array([[cl["shared"], cl["spec_A"] + cl["spec_B"]], [cBC["shared"], cBC["spec_A"] + cBC["spec_B"]]])
    print(f"boundary {rd}: B/D active early {ce['active']} (raster 60 +/- 6), late {cl['active']} (raster 70 +/- 7); late B/D shared {cl['shared']}/{cl['active']} = {fl:.2f} "
          f"vs late B/C {cBC['shared']}/{cBC['active']} = {fBC:.2f}; chi-square P = {stats.chi2_contingency(tab, correction=False)[1]:.2g}")

**Result: reproduced, resting on the boundary assumption.** Late in bird 1, 40 of 52 neurons active in B or D are shared (0.77; boundary reading a) against 8 of 92 for B and C (0.09; chi-square P = 7.3e-17); with reading (b), 28 of 38. The raster sizes are 46 and 52 (reading a) or 60 and 38 (reading b), against 60 and 70.

## motif-strategy-shared (blocked; bird 4 part shown)

The claim needs bird 3's syllables A to E, which the dataset does not label (data check), and depends on ed8-bird3-phase, which is blocked. Bird 4's part is shown: rhythmic protosyllable-stage neurons and, for each of its syllables, the number of neurons shared with another syllable (any pair of its labelled syllables, all ages).

In [ ]:
G = J[J.bird == "to5640"]
print("bird 4 rhythmic protosyllable-stage neurons:", int(G[G.code == 2].rhythmic_aa.sum()), "of", int((G.code == 2).sum()))
b4_types = sorted({t for k in G.key for t in hvc.syllables(NK[k])[3] if t})
sig = {}
for k in G.key:
    for t_ in b4_types:
        r = hvc.type_test(NK[k], {t_})
        sig[(k, t_)] = r is not None and r["p"] < hvc.ALPHA
shared_with_other = {t_: sum(1 for k in G.key if sig[(k, t_)] and any(sig[(k, u)] for u in b4_types if u != t_)) for t_ in b4_types}
print("bird 4 syllables:", b4_types, "; neurons significant for the syllable and at least one other:", shared_with_other)

## ed9-bird4-rhythms

> In bird 4's protosyllable stage, 11 of 18 neurons are rhythmic; 5 of these burst at 10 Hz and 6 at 5 Hz, on alternate cycles of the 10 Hz rhythm.

In [ ]:
G = J[(J.bird == "to5640") & (J.code == 2)]
print(f"bird 4 protosyllable-stage neurons: {len(G)} (expected 18 +/- 1.8)")
for col in ("rhythmic_aa", "rhythmic_bb"):
    R = G[G[col] == 1]
    per = R[f"period_{col[-2]}"]
    a10 = int(((per - 0.1).abs() < (per - 0.2).abs()).sum()); a5 = len(R) - a10
    f10 = int(((R.spec_freq - 10).abs() < (R.spec_freq - 5).abs()).sum()); f5 = len(R) - f10
    print(f"{col}: rhythmic {len(R)} (expected 11 +/- 1.1); (a) period nearer 100 ms: {a10}, nearer 200 ms: {a5}; (b) spectral peak nearer 10 Hz: {f10}, nearer 5 Hz: {f5} (expected 5 and 6)")
    print("   periods (ms):", (per * 1000).round(0).tolist(), " spectral peaks (Hz):", R.spec_freq.round(1).tolist())

**Result: not reproduced.** Bird 4 has 18 protosyllable-stage neurons, as expected, but 8 are rhythmic (11 +/- 1.1), 3 with a period near 100 ms and 5 near 200 ms (5 and 6 expected); the spectral peaks give the same split.

## ed4j-bias

> Specific neurons have a bias near one (0.96 +/- 0.011), while shared neurons have a broad range of bias (0.28 +/- 0.22).

Bias = 1 - min(r1, r2)/max(r1, r2), r the peak of each type's histogram in the test window (for the non-significant type of a specific neuron, its maximum in the same window). Readings: (a) the pairs above (early and late; bird 6 with each of its pairs in turn, the pair with the most shared neurons shown), (b) bird 1 beta/gamma and bird 2 alpha/beta.

In [ ]:
def bias_of(keys):
    T = pd.concat([PT[k] for k in keys])
    T = T[T.cls != "none"].dropna(subset=["peak_A", "peak_B"])
    T["bias"] = hvc.bias(T.peak_A.values, T.peak_B.values)
    return T
for lab, ks in (("(a) all pairs", read_a + [b6s[-1]]), ("(a') all pairs, bird 6 pair with fewest shared", read_a + [b6s[0]]),
                ("(b) bird 1 beta/gamma and bird 2 alpha/beta", ["b1_bg_early", "b1_BC_late", "b2_ab_early", "b2_ab_late"]),
                ("(b') early only", ["b1_bg_early", "b2_ab_early"])):
    T = bias_of(ks)
    sp = T[T.cls.str.startswith("specific")].bias; sh = T[T.cls == "shared"].bias
    print(f"{lab}: specific {sp.mean():.3f} +/- {sp.std():.3f} (n = {len(sp)}; expected 0.96 +/- 0.011), shared {sh.mean():.3f} +/- {sh.std():.3f} (n = {len(sh)}; expected 0.28 +/- 0.22); "
          f"specific bias range {sp.min():.2f} to {sp.max():.2f}")

**Result: partially reproduced.** Shared neurons have a broad bias, 0.282 +/- 0.228 over all pairs (0.28 +/- 0.22), and specific neurons a bias near one, 0.973 +/- 0.075, just outside 0.96 +/- 0.011. Over bird 1 beta/gamma and bird 2 alpha/beta (reading b) the means are 0.980 and 0.393; over their early periods only, 0.967 and 0.394, both inside.

## ed5-shared-ra-x-proportion

> Shared neurons are found in similar proportions among HVCRA and HVCX neurons (19% and 28%; P = 0.08).

A claim of absence: chi-square (2 x 2, cell type by shared or not, no continuity correction; Fisher's exact test when an expected count is below 5). Denominators: (a) the shared and specific neurons of the pairs, by cell type; (b) all juvenile neurons of the cell type in birds 1 to 6. A neuron that is shared in any pair counts as shared.

In [ ]:
allp = pd.concat([PT[k] for k in read_a + [b6s[-1]]])
allp = allp[allp.cls != "none"]
per_neuron = allp.groupby("key").cls.agg(lambda x: "shared" if (x == "shared").any() else "specific")
tabs_ = {}
for lab, base in (("(a) shared/specific neurons of the pairs", per_neuron.index),
                  ("(b) all juvenile neurons of birds 1 to 6", J[J.bird.isin(["to3965", "to3779", "to3017", "to5640", "to3396", "to2309"])].key)):
    rows = []
    for ct in ("HVC-RA", "HVC-X"):
        ks = [k for k in base if U.loc[k, "tab"] == ct]
        s_ = sum(1 for k in ks if per_neuron.get(k) == "shared")
        rows.append([s_, len(ks) - s_])
    tab = np.array(rows)
    ex = stats.contingency.expected_freq(tab)
    p = stats.fisher_exact(tab)[1] if (ex < 5).any() else stats.chi2_contingency(tab, correction=False)[1]
    print(f"{lab}: HVCRA {tab[0,0]}/{tab[0].sum()} = {tab[0,0]/tab[0].sum()*100:.0f}%, HVCX {tab[1,0]}/{tab[1].sum()} = {tab[1,0]/tab[1].sum()*100:.0f}% (expected 19% and 28%); P = {p:.3f}")

**Result: reproduced.** The shared proportions do not differ significantly between cell types: 24% of HVCRA and 34% of HVCX among the neurons of the pairs (P = 0.116; reading a) and 14% and 20% of all juvenile neurons of birds 1 to 6 (P = 0.085; reading b), against 19% and 28% (P = 0.08).

## ed2mo-ra-x-similar

> HVCRA and HVCX neurons develop alike; of 14 comparisons of locking, rhythmicity and period across stages, only the motif-stage period differs between cell types.

Juvenile stages only (12 comparisons at most; the adult stage is not in the dataset). Locking and rhythmicity: chi-square (Fisher when an expected count is below 5); period: readings (a) rank-sum, (b) chi-square on periods split at the stage median. Bonferroni over the comparisons made.

In [ ]:
rows = []
for code in range(1, 5):
    G = J[(J.code == code) & J.tab.isin(["HVC-RA", "HVC-X"])]
    for meas in ("locked_b", "rhythmic_aa"):
        tab = pd.crosstab(G.tab, G[meas]).reindex(index=["HVC-RA", "HVC-X"], columns=[0, 1], fill_value=0).values
        if tab.sum(axis=0).min() == 0 or tab.sum(axis=1).min() == 0:
            continue
        ex = stats.contingency.expected_freq(tab)
        p = stats.fisher_exact(tab)[1] if (ex < 5).any() else stats.chi2_contingency(tab, correction=False)[1]
        rows.append(dict(stage=code, measure=meas, RA=f"{tab[0,1]}/{tab[0].sum()}", X=f"{tab[1,1]}/{tab[1].sum()}", p_a=p, p_b=p))
    R = G[G.rhythmic_aa == 1]
    ra, xx = R[R.tab == "HVC-RA"].period_a, R[R.tab == "HVC-X"].period_a
    if len(ra) >= 1 and len(xx) >= 1:
        pa = stats.ranksums(ra, xx).pvalue
        med = R.period_a.median()
        tab = np.array([[(ra > med).sum(), (ra <= med).sum()], [(xx > med).sum(), (xx <= med).sum()]])
        ex = stats.contingency.expected_freq(tab)
        pb_ = stats.fisher_exact(tab)[1] if (ex < 5).any() else stats.chi2_contingency(tab, correction=False)[1]
        rows.append(dict(stage=code, measure="period", RA=f"{ra.median()*1000:.0f} ms (n={len(ra)})", X=f"{xx.median()*1000:.0f} ms (n={len(xx)})", p_a=pa, p_b=pb_))
M = pd.DataFrame(rows)
M["sig_a"] = M.p_a < 0.05 / len(M); M["sig_b"] = M.p_b < 0.05 / len(M)
print(M.round(4).to_string(index=False))
print(f"comparisons: {len(M)}; significant after Bonferroni, period reading (a): {M[M.sig_a][['stage','measure']].values.tolist()}; reading (b): {M[M.sig_b][['stage','measure']].values.tolist()}")

**Result: reproduced (juvenile stages only).** Of the 11 juvenile comparisons that can be made (subsong has no rhythmic HVCRA neuron), only the motif-stage period differs between HVCRA and HVCX after Bonferroni correction (494 against 413 ms; P = 0.0010 rank-sum, 0.0003 chi-square). The adult comparisons are not possible.

## ed2st-coverage

> Recorded bursts cover most of each juvenile syllable; 17 of 20 analysed syllables have over 90% covered (one only 73%), and every syllable's coverage lies within the range expected from random sampling of uniformly placed bursts.

The 20 syllables are not listed in the paper. Here every syllable type with a raster in the paper's figures for birds 1, 2 and 4 (and the protosyllable of each) is analysed, per period: bird 1 alpha (56-59), beta and gamma (62-72), B, C (81-112) and B, D (late, 80-112); bird 2 alpha (57-64), alpha and beta (65-72, 73-84); bird 4 alpha (48-49), b, c, d early and late (boundary a). A syllable is kept when more than 10 neurons have a burst window in it. Thresholds 2 Hz in the protosyllable stage and 10 Hz later.

In [ ]:
cov_list = [("b1 alpha 56-59", "to3965", 56, 59, {"a"}, "protosyll"), ("b1 beta 62-72", "to3965", 62, 72, {"b", "B"}, "older"),
            ("b1 gamma 62-72", "to3965", 62, 72, {"c", "C"}, "older"), ("b1 B 81-112", "to3965", 81, 112, {"B"}, "older"),
            ("b1 C 81-112", "to3965", 81, 112, {"C"}, "older"), ("b1 D 80-112", "to3965", 80, 112, {"D"}, "older"),
            ("b1 B 70-79", "to3965", 70, 79, {"B"}, "older"), ("b1 D 70-79", "to3965", 70, 79, {"D"}, "older"),
            ("b2 alpha 57-64", "to3779", 57, 64, {"a"}, "protosyll"), ("b2 alpha 65-72", "to3779", 65, 72, {"a"}, "older"),
            ("b2 beta 65-72", "to3779", 65, 72, {"b"}, "older"), ("b2 alpha 73-84", "to3779", 73, 84, {"a"}, "older"),
            ("b2 beta 73-84", "to3779", 73, 84, {"b"}, "older"), ("b4 alpha 48-49", "to5640", 48, 49, {"a"}, "protosyll"),
            ("b4 b 51-58", "to5640", 51, 58, {"b"}, "older"), ("b4 c 51-58", "to5640", 51, 58, {"c"}, "older"),
            ("b4 d 51-58", "to5640", 51, 58, {"d"}, "older"), ("b4 b 59-67", "to5640", 59, 67, {"b"}, "older"),
            ("b4 c 59-67", "to5640", 59, 67, {"c"}, "older"), ("b4 d 59-67", "to5640", 59, 67, {"d"}, "older")]
rng_cov = np.random.default_rng([hvc.SEED, 7])
rows = []
for name, b, a0, a1, labs_, st in cov_list:
    pls, nbs = [], []
    for k in keys_of(b, a0, a1):
        p_, nb_ = hvc.coverage_patches(NK[k], labs_, hvc.COV_THRESH[st])
        if p_:
            pls.append(p_); nbs.append(nb_)
    if not nbs:
        rows.append(dict(syllable=name, neurons=0)); continue
    nb_ = int(np.median(nbs))
    pls = [[p[:nb_] if p.size >= nb_ else np.r_[p, np.zeros(nb_ - p.size, bool)] for p in ps] for ps in pls]
    cf, lo, hi = hvc.coverage(pls, nb_, rng_cov)
    rows.append(dict(syllable=name, neurons=len(pls), window_ms=nb_, covered=round(cf * 100, 1), ci_lo=round(lo * 100, 1), ci_hi=round(hi * 100, 1),
                     within=bool(lo <= cf <= hi)))
C = pd.DataFrame(rows)
print(C.to_string(index=False))
K = C[C.neurons > hvc.COV_MIN_NEURONS]
print(f"syllables with more than 10 neurons bursting: {len(K)} (paper 20); over 90% covered: {int((K.covered > 90).sum())} (expected 17 +/- 1.7 of 20); "
      f"lowest {K.covered.min():.1f}% (expected 73 +/- 7.3%); within the surrogate interval: {int(K.within.sum())} of {len(K)}")

**Result: partially reproduced.** Of the 19 syllables analysed here with more than 10 neurons bursting (the paper's list of 20 is not given), 17 are covered above 90% (17 +/- 1.7 expected) and the lowest is 79.8% (73 +/- 7.3%). 18 of 19 lie within their surrogate interval; bird 1's B at 81 to 112 dph (93.9%) is below its interval (94.5 to 100%).

## The model (claims model-*)

These claims rest on the model code of the paper's Supplementary Information, not on the recordings (`data: elsewhere`). `hvcmodel.py` ports it to Python line by line (its docstring gives the code lines and Methods passages for each parameter). MATLAB's random generator cannot be reproduced, so the runs use NumPy's generator with the code's seeds (9038 alternating, 1009 bout-onset); ten further seeds are run as a variant where the criteria ask for it. Times are in 10 ms steps; latency 1 is the step of the seed input.

In [ ]:
import hvcmodel as H
t = time.time()
A = H.run_alternating(9038); SA = A["snapshots"]; seeds = A["seeds"]
print(f"alternating run (seed 9038): {time.time()-t:.0f} s; snapshots {list(SA)}")
def sscount(sn, sd):
    return H.shared_specific(H.find_latency(sn["xdyn"], sn["types"]), sd)

### model-fig5a-subsong-decay

> In the model before learning, activating the seed neurons produces a briefly propagating sequence of activity that decays within tens of milliseconds.

"Returns to the level" is taken as: from some lag of 10 steps or less on, the mean number of active non-seed neurons stays within 15% of the baseline (steps with no seed input in the 10 steps before), over all seed activations (isolated activations, with no other activation 10 steps before or 20 after, are too few for a stable mean and are shown alongside). The 15% band is a standard choice.

In [ ]:
def decay_report(Rn, label):
    d = H.subsong_decay(Rn["snapshots"]["subsong"]["xdyn"], Rn["snapshots"]["subsong"]["is_onset"], Rn["seeds"])
    base = d["baseline"]
    tr = d["all"]
    back = next((l for l in range(1, 11) if all(abs(tr[j] - base) <= 0.15 * base for j in range(l, 11))), None)
    print(f"{label}: onsets {d['n_all']} (isolated {d['n_isolated']}), baseline {base:.2f} active non-seed neurons per step; "
          f"after onset (lags 0-10, all onsets): {np.round(tr[:11], 2).tolist()}; isolated onsets: {np.round(d['isolated'][:11], 2).tolist()}; "
          f"within 15% of baseline from lag {back} on")
    return back
decay_report(A, "seed 9038")
VAR = {}
for sd in range(1, 11):
    VAR[sd] = H.run_alternating(sd)
    decay_report(VAR[sd], f"seed {sd}")

**Result: reproduced.** Before learning, the mean number of active non-seed neurons rises to 5.74 two steps after a seed activation and is within 15% of the baseline of 4.78 from 3 steps (30 ms) on. In ten further seeds it is back within 15% from 2 to 9 steps.

### model-fig5b-chain

> Rhythmic activation of the seed neurons with learning grows a feedforward chain that supports stable propagation of activity through the protosyllable.

In [ ]:
ls_ = H.find_latency(SA["subsong"]["xdyn"], SA["subsong"]["types"])
l5 = H.find_latency(SA[500]["xdyn"], SA[500]["types"])
c_sub, c_500 = H.shared_specific(ls_, seeds), H.shared_specific(l5, seeds)
lat500 = H.latencies(l5, seeds, [0])
print(f"participating non-seed neurons: subsong {c_sub['participating']}, iteration 500 {c_500['participating']}")
print("iteration 500, non-seed neurons per most common latency 1..10:", [int((lat500 == k).sum()) for k in range(1, 11)],
      "; seed neurons fire at latency 1 (the input step)")
for sd in range(1, 11):
    S5 = VAR[sd]["snapshots"]
    l = H.latencies(H.find_latency(S5[500]["xdyn"], S5[500]["types"]), VAR[sd]["seeds"], [0])
    print(f"  seed {sd}: participating subsong {sscount(S5['subsong'], VAR[sd]['seeds'])['participating']}, it500 {len(l)}; latencies 2-10 covered: {set(range(2, 11)) <= set(l.astype(int))}")

**Result: reproduced.** After 500 protosyllable iterations 90 non-seed neurons participate (26 in subsong), and their most common latencies cover every step from 2 to 10 of the cycle; step 1 is the seed input step, held by the seed neurons. Steps 2 to 10 are all covered in 7 of 10 further seeds.

### model-fig5cd-split

> Activating two halves of the seed neurons on alternate cycles splits the chain into two daughter chains; during splitting there are neurons specific to each syllable and shared neurons active at the same latency in both, and the fraction of shared neurons decreases.

In [ ]:
for label, Rn in [("seed 9038", A)] + [(f"seed {sd}", VAR[sd]) for sd in range(1, 11)]:
    c1, c2 = sscount(Rn["snapshots"][992], Rn["seeds"]), sscount(Rn["snapshots"][2500], Rn["seeds"])
    print(f"{label}: it992 specific {c1['specific']}, shared {c1['shared']} (same latency {c1['shared_same_lat']}), shared fraction {c1['shared_fraction']:.2f}; "
          f"it2500 specific {c2['specific']}, shared {c2['shared']}, fraction {c2['shared_fraction']:.2f}")

**Result: reproduced.** At iteration 992 there are specific neurons of both types (1 and 1) and 81 shared neurons, all with the same latency in both types; at iteration 2,500 the shared fraction has fallen from 0.98 to 0. In ten further seeds the shared fraction falls in every one (to 0 to 0.16), and 8 of 10 have specific neurons of both types at 992.

### model-fig5e-latency-broadens

> In the model, the distribution of burst latencies broadens from subsong through the protosyllable stage to chain splitting.

In [ ]:
def breadth(Rn):
    S_, sd = Rn["snapshots"], Rn["seeds"]
    a = H.latencies(H.find_latency(S_["subsong"]["xdyn"], S_["subsong"]["types"]), sd)
    b = H.latencies(H.find_latency(S_[500]["xdyn"], S_[500]["types"]), sd, [0])
    c = np.r_[H.latencies(H.find_latency(S_[992]["xdyn"], S_[992]["types"]), sd), H.latencies(H.find_latency(S_[2500]["xdyn"], S_[2500]["types"]), sd)]
    return [(len(x), stats.iqr(x), x.std()) for x in (a, b, c)]
for label, Rn in [("seed 9038", A)] + [(f"seed {sd}", VAR[sd]) for sd in range(1, 11)]:
    r = breadth(Rn)
    print(f"{label}: n {[x[0] for x in r]}; (a) IQR subsong/protosyllable/splitting {[round(x[1], 2) for x in r]}; (b) s.d. {[round(x[2], 2) for x in r]} (steps)")

**Result: reproduced (s.d. reading).** The s.d. of latencies grows from 2.18 (subsong) to 2.27 (iteration 500) and 2.41 steps (splitting). The IQR grows from 1 to 4 steps and then stays at 4, so the IQR reading does not show the second step.

### model-ed10gh-period

> In the model, the average period of rhythmic bursting increases during chain splitting as specific neurons burst only on alternate cycles of the protosyllable rhythm.

Readings: (a) autocorrelation peak of each participating non-seed neuron's activity in one iteration (lags 80 ms to the iteration's length), (b) most common inter-burst interval.

In [ ]:
def periods_at(sn, sd):
    lat = H.find_latency(sn["xdyn"], sn["types"])
    fd = np.array([L["fire_dur"] for L in lat]); ns = np.ones(H.N, bool); ns[sd] = False
    part = fd.any(0) & ns; spec = (fd.sum(0) == 1) & ns
    pa, pb = H.period_autocorr(sn["xdyn"]), H.period_ibi(sn["xdyn"])
    return part, spec, pa, pb
for it in (500, 992, 2500):
    part, spec, pa, pb = periods_at(SA[it], seeds)
    print(f"iteration {it}: (a) mean period participating {np.nanmean(pa[part]):.1f} ms (n={part.sum()}), specific {np.nanmean(pa[spec]) if spec.any() else np.nan:.1f} ms (n={spec.sum()}); "
          f"(b) {np.nanmean(pb[part]):.1f} ms, specific {np.nanmean(pb[spec]) if spec.any() else np.nan:.1f} ms")

**Result: reproduced.** The mean period of participating neurons is 100 ms at iteration 500, 102.4 ms at 992 and 200 ms at 2,500, where the 75 specific neurons all have a period of 200 ms; both readings agree.

### model-ed10ad-bout-onset

> Adding bout-onset seed neurons makes the protosyllable chain split into one daughter chain active only at bout onset and another active only on later syllables.

In [ ]:
t = time.time()
B = H.run_bout_onset(1009)
print(f"bout-onset run (seed 1009): {time.time()-t:.0f} s")
for it in (5, 100, 130, 600):
    c = sscount(B["snapshots"][it], B["seeds"])
    print(f"iteration {it}: specific [bout-onset, later] {c['specific']}, shared {c['shared']}, shared fraction {c['shared_fraction']:.2f}")

**Result: reproduced.** At iteration 600 there are 51 neurons specific to the bout-onset syllable and 39 to later syllables, with no shared neuron, against a shared fraction of 0.60 at iteration 130.

### model-ed10eh-motif

> Dividing the seed neurons into three groups activated in turn splits the protosequence into three non-overlapping daughter sequences, simulating the motif strategy.

Reading (a) of the criterion (the alternating run's iteration counts and thresholds; the GitHub code is not pinned, so reading (b) is not run). Seed groups are pulsed in turn every 100 ms (`hvcmodel.run_motif`).

In [ ]:
t = time.time()
M = H.run_motif(9038)
print(f"motif run (seed 9038): {time.time()-t:.0f} s")
for it in (500, 992, 2500):
    sn = M["snapshots"][it]
    lat = H.find_latency(sn["xdyn"], sn["types"])
    c = H.shared_specific(lat, M["seeds"])
    fd = np.array([L["fire_dur"] for L in lat]); ns = np.ones(H.N, bool); ns[M["seeds"]] = False
    print(f"iteration {it}: specific per group {c['specific']}, in two or more groups {c['shared']} (same latency {c['shared_same_lat']}), "
          f"in all three {int(((fd.sum(0) == 3) & ns).sum())}; shared fraction {c['shared_fraction']:.2f}")

**Result: partially reproduced.** After 2,500 iterations each of the three seed groups has specific neurons (32, 11 and 17) and no neuron participates in all three, but 13 neurons (shared fraction 0.18) still participate in two groups, so the sequences are not fully non-overlapping. The iteration counts and thresholds are an assumption.

### model-ed10ik-new-chain

> With stronger seed input, bout-onset seed neurons grow a new syllable chain at bout onset while the protosyllable chain stays intact, without bout-onset differentiation.

In [ ]:
t = time.time()
Cn = H.run_new_chain(1009)
print(f"new-chain run (seed 1009): {time.time()-t:.0f} s")
for it in (5, 100, 130, 600):
    c = sscount(Cn["snapshots"][it], Cn["seeds"])
    print(f"iteration {it}: specific [bout-onset, protosyllable] {c['specific']}, shared {c['shared']}, shared fraction {c['shared_fraction']:.2f}")
sn = Cn["snapshots"][600]
print("iteration 600, fraction of protosyllable inputs on which each protosyllable seed neuron fires:", H.seed_reliability(sn["xdyn"], sn["ind_psyl"], Cn["psyl_seeds"]).round(2).tolist())
sn = B["snapshots"][600]
print("same in the bout-onset differentiation run:", H.seed_reliability(sn["xdyn"], sn["ind_psyl"], B["psyl_seeds"]).round(2).tolist())

**Result: reproduced.** With the stronger seed input, at iteration 600 there are 34 neurons specific to the bout-onset element and 10 to the protosyllable, and each protosyllable seed neuron fires on every protosyllable input (1.0, against 0.67 in the bout-onset differentiation run).

## ed8-bird3-phase (blocked)

> In bird 3 (motif strategy) shared neurons burst at similar phases of the song rhythm in their two syllables (mean absolute phase difference 41 +/- 33.9 degrees, 8 neurons; smaller than shuffled, P = 0.013).

The test needs bird 3's phase-segmented syllables labelled A to E (Extended Data Fig. 8). In the dataset, bird 3's selected segments carry no labels except in one neuron:

In [ ]:
G = [k for k in U.index if U.loc[k, "bird"] == "to3017"]
lab3 = {k: collections.Counter(hvc.syllables(NK[k])[3]) for k in G}
print(f"bird 3 neurons: {len(G)}; with any labelled syllable: {sum(1 for c in lab3.values() if sum(v for t, v in c.items() if t) > 0)}")
for k, c in lab3.items():
    if sum(v for t, v in c.items() if t):
        print(" ", k, dict(c))
print("unlabelled selected segments in bird 3:", sum(c.get("", 0) for c in lab3.values()))

## ed5-shared-acoustic-difference (blocked)

The test computes eight Sound Analysis Pro features (refs 51 and 55) of the song in 1 ms steps after each burst of every shared neuron, then Fisher's discriminant and a d' trajectory with 1,000 label shuffles. The sound is only in the per-bird raw archives (to3965 alone is 12.5 GB compressed; birds 1 to 6 together 30 GB), and Sound Analysis Pro is Windows software that cannot be installed in the locked environment. No public implementation pinned by the paper reproduces its feature definitions. Not run.

## Summary

The outcomes are decided in the attempt record from the numbers printed above; this table repeats them.

In [ ]:
print(f"notebook run time {(time.time() - T0)/60:.1f} min; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

**Summary.**

| Claim | Outcome |
|---|---|
| counts-neurons-birds | reproduced (juvenile part) |
| ed1a-stage-age | reproduced |
| fig1-subsong-locking | not reproduced |
| fig1f-locking-increases | partially reproduced (juvenile only) |
| fig1f-rhythmic-protosyllable | partially reproduced |
| fig1fg-rhythm-increases | partially reproduced (juvenile only) |
| ed1c-burst-width-rate | blocked: data missing (adult neurons) |
| fig1de-bout-related | not reproduced |
| fig2b-protosequence | partially reproduced |
| ed2st-coverage | partially reproduced (assumed) |
| ed2p-probabilistic | reproduced |
| ed2qr-pairs-independent | partially reproduced |
| fig2c-latency-subsong-earlier | reproduced |
| fig2c-latency-broadens-later | reproduced |
| bird1-protosequence | reproduced |
| fig3g-bird1-shared-specific | reproduced (assumed) |
| ed4a-shared-latency-bird1 | partially reproduced (assumed) |
| ed3j-shared-ibi-specific-doubled | reproduced |
| fig3h-bird1-shared-decrease | partially reproduced (assumed) |
| all-birds-shared-specific | partially reproduced (assumed) |
| all-birds-shared-decrease | reproduced (assumed) |
| skipping-cycles | partially reproduced |
| fig4-bird2-before-beta | partially reproduced |
| fig4h-bird2-shared-early | partially reproduced (assumed) |
| fig4i-bird2-shared-decrease | partially reproduced (assumed) |
| ed4bd-shared-latency-other | not reproduced (assumed) |
| ed7-bird1-bd-shared-persist | reproduced (assumed) |
| ed8-bird3-phase | blocked: metadata insufficient |
| motif-strategy-shared | blocked: metadata insufficient |
| ed9-bird4-rhythms | not reproduced |
| ed4j-bias | partially reproduced (assumed) |
| ed5-shared-acoustic-difference | blocked: not runnable |
| ed5-shared-ra-x-proportion | reproduced (assumed) |
| ed2mo-ra-x-similar | reproduced (juvenile only) |
| model claims (8) | 7 reproduced, 1 partially reproduced (motif strategy) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The adult neurons (ref. 20) are not in the dataset, so every comparison with the adult stage is left out. The paper adjusted the shared/specific test window by eye; here a fixed rule stands in for it (an assumption), and the unadjusted result is shown. The syllable pairs for birds 5 and 6, the 20 coverage syllables and the early/late boundaries of bird 1's B/D and bird 4's pairs are not given in the paper and are chosen here. Bird 4's protosyllables are the dataset's `a` labels, not recomputed by phase segmentation. The model runs on NumPy's random generator, not MATLAB's.

**What was found about the dataset.** The files hold 1,125 neurons of 32 juvenile birds with spike times, syllable segments and labels; 1,116 pass the 5 s singing rule, one fewer than the paper's 1,117, and the per-stage counts (39, 135, 564, 378) match the paper's to within one. The neuron list has a stage, 'young adult' (66 neurons of bird 1, 90 to 112 dph), that the paper counts as motif. Two rows of the list share one name (to3563_2013-02-25_HVCp-1) and only one file exists for them. Bird 3's syllables carry no labels, so its motif-strategy analysis cannot be repeated. The labels use Latin letters (a to e for alpha to epsilon), which the data descriptor does not say.